In [3]:
!pip install datasets

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.26a0+c5e1555-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.14.0-py3.12.egg is depreca

In [4]:
import json, os, urllib.request
from datasets import load_dataset

BASE = "https://fever.ai/download/fever"
os.makedirs("fever_raw", exist_ok=True)
os.makedirs("fever_clean", exist_ok=True)

FILES = {
    "train": "train.jsonl",
    "labelled_dev": "shared_task_dev.jsonl",
    "unlabelled_test": "shared_task_test.jsonl",
}

REQUIRED_COLS = ["id", "verifiable", "label", "claim", "evidence"]

def download(url, out_path):
    if not os.path.exists(out_path):
        print("⬇️ downloading:", url)
        urllib.request.urlretrieve(url, out_path)

def normalize_example(ex: dict) -> dict:
    # Ensure all required columns exist in ALL splits
    for k in REQUIRED_COLS:
        if k not in ex:
            ex[k] = ""   # keep consistent type (string) across splits

    # id -> int if possible (keeps consistent int64)
    try:
        ex["id"] = int(ex["id"])
    except Exception:
        pass

    # evidence -> string ALWAYS (so Arrow schema is stable)
    # Some splits may already be "", keep it as string.
    if ex["evidence"] != "" and not isinstance(ex["evidence"], str):
        ex["evidence"] = json.dumps(ex["evidence"], ensure_ascii=False)

    # If evidence is a list/dict inside a string? leave it.
    if isinstance(ex["evidence"], (list, dict)):
        ex["evidence"] = json.dumps(ex["evidence"], ensure_ascii=False)

    # Make other fields strings (FEVER often uses strings)
    for k in ["verifiable", "label", "claim"]:
        if ex[k] is None:
            ex[k] = ""
        if not isinstance(ex[k], str):
            ex[k] = str(ex[k])

    return ex

def clean_jsonl(in_path, out_path):
    with open(in_path, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            ex = json.loads(line)
            ex = normalize_example(ex)
            fout.write(json.dumps(ex, ensure_ascii=False) + "\n")

raw_paths = {}
clean_paths = {}

for split, fname in FILES.items():
    url = f"{BASE}/{fname}"
    raw = os.path.join("fever_raw", fname)
    clean = os.path.join("fever_clean", fname)

    download(url, raw)
    clean_jsonl(raw, clean)

    raw_paths[split] = raw
    clean_paths[split] = clean

# ✅ Now schemas match across all files, so this won't crash
ds = load_dataset("json", data_files=clean_paths)

train = ds["train"]
dev   = ds["labelled_dev"]
test  = ds["unlabelled_test"]

print("sizes:", len(train), len(dev), len(test))
print("train columns:", train.column_names)
print("test columns :", test.column_names)
print("sample train evidence (string):", train[0]["evidence"][:200])
print("sample test label/verifiable:", test[0]["label"], test[0]["verifiable"])

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using custom data configuration default-5c54e85a62815069


Extracting data files: 100%|██████████| 3/3 [00:00<00:00, 1899.88it/s]


Dataset json downloaded and prepared to /root/.cache/huggingface/datasets/json/default-5c54e85a62815069/0.0.0/ac0ca5f5289a6cf108e706efcf040422dbbfa8e658dee6a819f20d76bb84d26b. Subsequent calls will reuse this data.


100%|██████████| 3/3 [00:00<00:00, 731.01it/s]

sizes: 145449 19998 19998
train columns: ['id', 'verifiable', 'label', 'claim', 'evidence']
test columns : ['id', 'verifiable', 'label', 'claim', 'evidence']
sample train evidence (string): [[[92206, 104971, "Nikolaj_Coster-Waldau", 7], [92206, 104971, "Fox_Broadcasting_Company", 0]]]
sample test label/verifiable:  


In [5]:
import json

def flatten_fever_evidence(evidence):
    """
    Handles both
    string and nested list formats
    Returns: list of (page_title, sentence_id)
    """

    if not evidence:
        return []

    # If stored as JSON string, decode it
    if isinstance(evidence, str):
        evidence = json.loads(evidence)

    out = []

    for ev_set in evidence:
        for ev in ev_set:
            if len(ev) >= 4:
                page = ev[2]
                sent_id = ev[3]
                out.append((page, sent_id))

    return out


print(flatten_fever_evidence(train[0]["evidence"]))


[('Nikolaj_Coster-Waldau', 7), ('Fox_Broadcasting_Company', 0)]


In [6]:
# ============================================================
# FEVER Route-2 (Proper, Publishable) - FULL PYTHON PIPELINE
# - Downloads FEVER wiki-pages dump (zip)
# - Unzips it
# - Builds SQLite DB: (title, line_id) -> sentence
# - Loads FEVER train/dev/test JSONL (your cleaned dataset is ok too)
# - Extracts evidence pairs and fetches exact evidence text offline
# ============================================================

import os
import re
import json
import glob
import time
import sqlite3
import zipfile
import urllib.request
from typing import List, Tuple, Optional

# Optional but nice progress bar
try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# -----------------------------
# Config (change base dir if you want)
# -----------------------------
BASE_DIR = "fever_data"
WIKI_ZIP_PATH = os.path.join(BASE_DIR, "wiki-pages.zip")
WIKI_DIR = os.path.join(BASE_DIR, "wiki-pages")
DB_PATH = os.path.join(BASE_DIR, "fever_wiki.sqlite")

FEVER_BASE_URL = "https://fever.ai/download/fever"
WIKI_ZIP_URL = f"{FEVER_BASE_URL}/wiki-pages.zip"

# FEVER dataset files (claims + labels + evidence)
TRAIN_URL = f"{FEVER_BASE_URL}/train.jsonl"
DEV_URL   = f"{FEVER_BASE_URL}/shared_task_dev.jsonl"
TEST_URL  = f"{FEVER_BASE_URL}/shared_task_test.jsonl"

TRAIN_PATH = os.path.join(BASE_DIR, "train.jsonl")
DEV_PATH   = os.path.join(BASE_DIR, "shared_task_dev.jsonl")
TEST_PATH  = os.path.join(BASE_DIR, "shared_task_test.jsonl")


# ============================================================
# Utils
# ============================================================

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def download_file(url: str, out_path: str, overwrite: bool = False):
    ensure_dir(os.path.dirname(out_path))
    if os.path.exists(out_path) and not overwrite:
        print(f"✅ Already exists: {out_path}")
        return
    print(f"⬇️ Downloading:\n  {url}\n  -> {out_path}")
    urllib.request.urlretrieve(url, out_path)
    print("✅ Downloaded")

def unzip_file(zip_path: str, out_dir: str, overwrite: bool = False):
    ensure_dir(out_dir)
    marker = os.path.join(out_dir, ".unzipped_ok")
    if os.path.exists(marker) and not overwrite:
        print(f"✅ Already unzipped: {out_dir}")
        return

    print(f"🧩 Unzipping:\n  {zip_path}\n  -> {out_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)

    # Some zips contain a nested folder "wiki-pages/"
    # We want WIKI_DIR to directly contain *.jsonl
    # If extraction made BASE_DIR/wiki-pages/wiki-pages/*.jsonl, fix it.
    nested = os.path.join(out_dir, "wiki-pages")
    if os.path.isdir(nested) and len(glob.glob(os.path.join(out_dir, "*.jsonl"))) == 0:
        # move nested contents up one level
        for fp in glob.glob(os.path.join(nested, "*")):
            base = os.path.basename(fp)
            os.replace(fp, os.path.join(out_dir, base))
        try:
            os.rmdir(nested)
        except Exception:
            pass

    with open(marker, "w") as f:
        f.write("ok\n")
    print("✅ Unzipped")

def parse_wiki_lines(lines_field: str) -> List[Tuple[int, str]]:
    """
    FEVER wiki dump format:
      lines: "0\tSentence...\n1\tSentence...\n..."
    Return list of (line_id, sentence)
    """
    out = []
    if not lines_field:
        return out
    for row in lines_field.split("\n"):
        row = row.strip()
        if not row:
            continue
        parts = row.split("\t", 1)
        if len(parts) != 2:
            continue
        try:
            line_id = int(parts[0])
        except Exception:
            continue
        sent = parts[1].strip()
        out.append((line_id, sent))
    return out


# ============================================================
# Build SQLite DB
# ============================================================

def build_sqlite_db(wiki_dir: str, db_path: str):
    ensure_dir(os.path.dirname(db_path))

    jsonl_files = sorted(glob.glob(os.path.join(wiki_dir, "*.jsonl")))
    if not jsonl_files:
        raise FileNotFoundError(
            f"No *.jsonl found in {wiki_dir}. Check unzip output."
        )

    print(f"📚 Found {len(jsonl_files)} wiki jsonl files")

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # Speed pragmas
    cur.execute("PRAGMA journal_mode=WAL;")
    cur.execute("PRAGMA synchronous=NORMAL;")
    cur.execute("PRAGMA temp_store=MEMORY;")

    cur.execute("""
    CREATE TABLE IF NOT EXISTS wiki_sentences (
        title TEXT NOT NULL,
        line_id INTEGER NOT NULL,
        sentence TEXT NOT NULL,
        PRIMARY KEY (title, line_id)
    )
    """)

    insert_sql = "INSERT OR REPLACE INTO wiki_sentences(title, line_id, sentence) VALUES (?, ?, ?)"

    total_inserted = 0
    batch = []

    iterator = jsonl_files
    if tqdm:
        iterator = tqdm(jsonl_files, desc="Indexing wiki-pages")

    t0 = time.time()
    for fp in iterator:
        with open(fp, "r", encoding="utf-8") as f:
            for line in f:
                ex = json.loads(line)
                title = ex.get("id")  # FEVER uses 'id' as title
                lines_field = ex.get("lines", "")
                if not title:
                    continue

                pairs = parse_wiki_lines(lines_field)
                for line_id, sent in pairs:
                    batch.append((title, line_id, sent))

                # Flush in chunks
                if len(batch) >= 50000:
                    cur.executemany(insert_sql, batch)
                    conn.commit()
                    total_inserted += len(batch)
                    batch.clear()

    if batch:
        cur.executemany(insert_sql, batch)
        conn.commit()
        total_inserted += len(batch)
        batch.clear()

    cur.execute("CREATE INDEX IF NOT EXISTS idx_title ON wiki_sentences(title)")
    conn.commit()
    conn.close()

    dt = time.time() - t0
    print(f"✅ SQLite built: {db_path}")
    print(f"   Inserted rows: {total_inserted:,}")
    print(f"   Time: {dt/60:.2f} min")


# ============================================================
# FEVER Evidence Helpers
# ============================================================

def flatten_fever_evidence(evidence) -> List[Tuple[str, int]]:
    """
    Handles both:
      - evidence as nested list
      - evidence as JSON string (if you cleaned it earlier)
    Returns list of (page_title, sentence_id)
    """
    if not evidence:
        return []

    if isinstance(evidence, str):
        evidence = json.loads(evidence)

    out = []
    for ev_set in evidence:
        for ev in ev_set:
            if isinstance(ev, list) and len(ev) >= 4:
                page = ev[2]
                sent_id = ev[3]
                if page is not None and sent_id is not None:
                    out.append((str(page), int(sent_id)))
    return out


class FeverWikiDB:
    def __init__(self, db_path: str):
        self.conn = sqlite3.connect(db_path)
        self.cur = self.conn.cursor()

    def get_sentence(self, title: str, line_id: int) -> Optional[str]:
        row = self.cur.execute(
            "SELECT sentence FROM wiki_sentences WHERE title=? AND line_id=?",
            (title, int(line_id))
        ).fetchone()
        return row[0] if row else None

    def close(self):
        self.conn.close()


def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)


# ============================================================
# MAIN: one-click setup + demo
# ============================================================

def setup_all():
    ensure_dir(BASE_DIR)

    # 1) Download FEVER splits
    download_file(TRAIN_URL, TRAIN_PATH)
    download_file(DEV_URL, DEV_PATH)
    download_file(TEST_URL, TEST_PATH)

    # 2) Download + unzip wiki dump
    download_file(WIKI_ZIP_URL, WIKI_ZIP_PATH)
    unzip_file(WIKI_ZIP_PATH, WIKI_DIR)

    # 3) Build sqlite DB (skip if exists)
    if os.path.exists(DB_PATH):
        print(f"✅ DB already exists: {DB_PATH}")
    else:
        build_sqlite_db(WIKI_DIR, DB_PATH)

def demo_one_example():
    wiki = FeverWikiDB(DB_PATH)

    # Read first example from train
    ex = next(iter_jsonl(TRAIN_PATH))
    claim = ex.get("claim")
    label = ex.get("label")
    evidence_pairs = flatten_fever_evidence(ex.get("evidence"))

    evidence_texts = []
    for page, sid in evidence_pairs:
        sent = wiki.get_sentence(page, sid)
        if sent:
            evidence_texts.append((page, sid, sent))

    print("\n================= DEMO (1 example) =================")
    print("CLAIM:", claim)
    print("LABEL:", label)
    print("\nEVIDENCE (page, sent_id, sentence):")
    for page, sid, sent in evidence_texts[:10]:
        print(f"- {page} [{sid}] {sent}")

    wiki.close()

if __name__ == "__main__":
    setup_all()
    demo_one_example()


✅ Already exists: fever_data/train.jsonl
✅ Already exists: fever_data/shared_task_dev.jsonl
✅ Already exists: fever_data/shared_task_test.jsonl
✅ Already exists: fever_data/wiki-pages.zip
✅ Already unzipped: fever_data/wiki-pages
✅ DB already exists: fever_data/fever_wiki.sqlite

================= DEMO (1 example) =================
CLAIM: Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.
LABEL: SUPPORTS

EVIDENCE (page, sent_id, sentence):
- Nikolaj_Coster-Waldau [7] He then played Detective John Amsterdam in the short-lived Fox television series New Amsterdam -LRB- 2008 -RRB- , as well as appearing as Frank Pike in the 2009 Fox television film Virtuality , originally intended as a pilot .	Fox television	Fox Broadcasting Company	New Amsterdam	New Amsterdam (TV series)	Virtuality	Virtuality (TV series)
- Fox_Broadcasting_Company [0] The Fox Broadcasting Company -LRB- often shortened to Fox and stylized as FOX -RRB- is an American English language commercial broadcast telev

In [1]:
import os

OUT_DIR = "fever_dense"
INDEX_PATH = os.path.join(OUT_DIR, "faiss.index")
META_PATH  = os.path.join(OUT_DIR, "passage_meta.jsonl")
STATE_PATH = os.path.join(OUT_DIR, "build_state.json")

for p in [INDEX_PATH, META_PATH, STATE_PATH]:
    if os.path.exists(p):
        print("Deleting:", p)
        os.remove(p)

print("✅ Deleted old corrupted artifacts. Now rebuild.")

Deleting: fever_dense/faiss.index
Deleting: fever_dense/passage_meta.jsonl
Deleting: fever_dense/build_state.json
✅ Deleted old corrupted artifacts. Now rebuild.


In [2]:
import os, re, json, sqlite3
from typing import List, Tuple, Optional

import numpy as np
import torch
import faiss
from sentence_transformers import SentenceTransformer

# =========================
# CONFIG
# =========================
DB_PATH = "fever_data/fever_wiki.sqlite"

OUT_DIR = "fever_dense"
os.makedirs(OUT_DIR, exist_ok=True)

INDEX_PATH = os.path.join(OUT_DIR, "faiss.index")
META_PATH  = os.path.join(OUT_DIR, "passage_meta.jsonl")
STATE_PATH = os.path.join(OUT_DIR, "build_state.json")

PASSAGE_SENTENCES = 3
MAX_SENT_CHARS = 350
MAX_PASSAGE_CHARS = 1200

EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMB_DIM = 384

HNSW_M = 32
BATCH_PASSAGES = 4096
COMMIT_EVERY = 20000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Safety switches ----
FORCE_REBUILD = False   # delete existing artifacts and rebuild from scratch
FORCE_RESUME  = False   # if True, try to resume build even if index exists


# =========================
# HELPERS
# =========================
def save_json(path: str, obj: dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: str):
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def append_jsonl(path: str, rows: List[dict]):
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def count_jsonl_lines(path: str) -> int:
    if not os.path.exists(path):
        return 0
    with open(path, "rb") as f:
        return sum(1 for _ in f)

def clean_sentence(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    if "\t" in s:
        s = s.split("\t", 1)[0].strip()
    if len(s) > MAX_SENT_CHARS:
        s = s[:MAX_SENT_CHARS].rsplit(" ", 1)[0]
    return s

def normalize_embeddings(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / norms


# =========================
# STREAM PASSAGES FROM SQLITE
# =========================
class WikiPassageStreamer:
    """
    Expects table:
      wiki_sentences(title TEXT, line_id INT, sentence TEXT)

    Creates passages of PASSAGE_SENTENCES consecutive sentences.
    """
    def __init__(self, db_path: str):
        self.conn = sqlite3.connect(db_path)
        self.cur = self.conn.cursor()

    def stream_passages(self, start_after: Optional[Tuple[str, int]] = None):
        """
        start_after = (title, line_id) meaning: resume AFTER this (title,line_id)
        """
        if start_after is None:
            q = """
            SELECT title, line_id, sentence
            FROM wiki_sentences
            ORDER BY title ASC, line_id ASC
            """
            params = ()
        else:
            last_title, last_line = start_after
            q = """
            SELECT title, line_id, sentence
            FROM wiki_sentences
            WHERE (title > ?) OR (title = ? AND line_id > ?)
            ORDER BY title ASC, line_id ASC
            """
            params = (last_title, last_title, int(last_line))

        cur_title = None
        buf: List[Tuple[int, str]] = []

        for title, line_id, sent in self.cur.execute(q, params):
            title = str(title)
            line_id = int(line_id)
            sent = clean_sentence(sent)
            if not sent:
                continue

            if cur_title is None:
                cur_title = title

            if title != cur_title:
                # flush old title
                for p in self._flush(cur_title, buf):
                    yield p
                cur_title = title
                buf = []

            buf.append((line_id, sent))

            # keep memory bounded
            if len(buf) >= 5000:
                emitted, rem = self._emit(buf, cur_title)
                for p in emitted:
                    yield p
                buf = rem

        if cur_title is not None and buf:
            for p in self._flush(cur_title, buf):
                yield p

    def _emit(self, buf: List[Tuple[int, str]], title: str):
        emitted = []
        i = 0
        n = len(buf)
        while i + PASSAGE_SENTENCES <= n:
            chunk = buf[i:i + PASSAGE_SENTENCES]
            start_line = chunk[0][0]
            end_line = chunk[-1][0]
            text = " ".join([c[1] for c in chunk]).strip()
            if len(text) > MAX_PASSAGE_CHARS:
                text = text[:MAX_PASSAGE_CHARS].rsplit(" ", 1)[0]
            emitted.append({
                "title": title,
                "start_line": start_line,
                "end_line": end_line,
                "text": text,
                # IMPORTANT: resume marker should be a real sentence line_id
                "resume_line_id": end_line,
            })
            i += PASSAGE_SENTENCES
        return emitted, buf[i:]

    def _flush(self, title: str, buf: List[Tuple[int, str]]):
        emitted, _ = self._emit(buf, title)
        return emitted

    def close(self):
        self.conn.close()


# =========================
# FAISS BUILD
# =========================
def create_hnsw_index() -> faiss.Index:
    print("🧠 Creating FAISS HNSW index (cosine via inner product + normalized vectors)")
    idx = faiss.IndexHNSWFlat(EMB_DIM, HNSW_M, faiss.METRIC_INNER_PRODUCT)
    idx.hnsw.efConstruction = 200
    idx.hnsw.efSearch = 128
    return idx

def build_faiss():
    if not os.path.exists(DB_PATH):
        raise FileNotFoundError(f"❌ DB not found: {DB_PATH}")

    # ---------- Rebuild mode ----------
    if FORCE_REBUILD:
        for p in [INDEX_PATH, META_PATH, STATE_PATH]:
            if os.path.exists(p):
                print("🧹 Removing:", p)
                os.remove(p)

    # ---------- Fast skip: already built ----------
    if os.path.exists(INDEX_PATH) and os.path.exists(META_PATH) and (not FORCE_RESUME) and (not FORCE_REBUILD):
        idx = faiss.read_index(INDEX_PATH)
        meta_n = count_jsonl_lines(META_PATH)
        print(f"✅ Found existing index + meta. index.ntotal={idx.ntotal:,} meta_rows={meta_n:,}")

        # consistency guard
        if meta_n == idx.ntotal and idx.ntotal > 0:
            print("✅ Artifacts look consistent. Skipping rebuild.")
            return
        else:
            print("⚠️ index/meta mismatch. Will attempt to resume/build to fix.")
            # continue into resume path

    # ---------- Resume state ----------
    state = load_json(STATE_PATH) or {}
    resume_after = state.get("resume_after")
    resume_tuple = None
    if resume_after and "title" in resume_after and "line_id" in resume_after:
        resume_tuple = (resume_after["title"], int(resume_after["line_id"]))
        print("🔁 Resuming after:", resume_tuple)
    else:
        print("ℹ️ No resume state found. Starting from beginning.")

    # ---------- Load or create FAISS index ----------
    if os.path.exists(INDEX_PATH) and not FORCE_REBUILD:
        print(f"✅ Loading existing FAISS index: {INDEX_PATH}")
        idx = faiss.read_index(INDEX_PATH)
    else:
        idx = create_hnsw_index()

    print("📌 Current vectors:", f"{idx.ntotal:,}")

    # ---------- Ensure meta exists ----------
    if not os.path.exists(META_PATH):
        open(META_PATH, "w", encoding="utf-8").close()

    # ---------- Meta vs index guard ----------
    meta_lines = count_jsonl_lines(META_PATH)
    if meta_lines != idx.ntotal:
        print(f"⚠️ META/INDEX mismatch: meta_rows={meta_lines:,} vs index.ntotal={idx.ntotal:,}")
        print("   This usually means a partial save happened. Resume will append new meta rows from now on.")
        # We do NOT attempt to “repair” old rows automatically here.

    # ---------- Embedder ----------
    print(f"🔧 Loading SBERT: {EMB_MODEL} on {DEVICE}")
    embedder = SentenceTransformer(EMB_MODEL, device=DEVICE)
    if DEVICE == "cuda":
        embedder = embedder.half()

    streamer = WikiPassageStreamer(DB_PATH)

    buffer = []
    last_seen = resume_tuple  # track as (title, line_id)
    added_since_save = 0

    def flush():
        nonlocal buffer, last_seen, added_since_save, idx
        if not buffer:
            return

        texts = [p["text"] for p in buffer]

        emb = embedder.encode(
            texts,
            batch_size=128 if DEVICE == "cuda" else 32,
            convert_to_numpy=True,
            normalize_embeddings=False,
            show_progress_bar=False
        ).astype("float32")

        emb = normalize_embeddings(emb)

        before = idx.ntotal
        idx.add(emb)
        n_added = idx.ntotal - before
        added_since_save += n_added

        meta_rows = []
        for i, p in enumerate(buffer):
            meta_rows.append({
                "vector_id": int(before + i),
                "title": p["title"],
                "start_line": int(p["start_line"]),
                "end_line": int(p["end_line"]),
                "text": p["text"]
            })
        append_jsonl(META_PATH, meta_rows)

        # update resume marker using a REAL sentence line_id
        last_p = buffer[-1]
        last_seen = (last_p["title"], int(last_p["resume_line_id"]))
        save_json(STATE_PATH, {"resume_after": {"title": last_seen[0], "line_id": int(last_seen[1])}})

        if added_since_save >= COMMIT_EVERY:
            print(f"💾 Saving index... ntotal={idx.ntotal:,}")
            faiss.write_index(idx, INDEX_PATH)
            added_since_save = 0

        buffer = []

    print("🚀 Building/Resuming FAISS from SQLite wiki...")
    for p in streamer.stream_passages(start_after=resume_tuple):
        buffer.append(p)

        if len(buffer) >= BATCH_PASSAGES:
            flush()
            if (idx.ntotal % 200000) < BATCH_PASSAGES:
                print(f"✅ Progress: {idx.ntotal:,} vectors")

    flush()

    print("💾 Final save...")
    faiss.write_index(idx, INDEX_PATH)
    if last_seen:
        save_json(STATE_PATH, {"resume_after": {"title": last_seen[0], "line_id": int(last_seen[1])}})
    streamer.close()

    print("✅ DONE")
    print("Index:", INDEX_PATH)
    print("Meta :", META_PATH)
    print("Total vectors:", f"{idx.ntotal:,}")


if __name__ == "__main__":
    build_faiss()

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


ℹ️ No resume state found. Starting from beginning.
🧠 Creating FAISS HNSW index (cosine via inner product + normalized vectors)
📌 Current vectors: 0
🔧 Loading SBERT: sentence-transformers/all-MiniLM-L6-v2 on cuda


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


🚀 Building/Resuming FAISS from SQLite wiki...
💾 Saving index... ntotal=20,480
💾 Saving index... ntotal=40,960
💾 Saving index... ntotal=61,440
💾 Saving index... ntotal=81,920
💾 Saving index... ntotal=102,400
💾 Saving index... ntotal=122,880
💾 Saving index... ntotal=143,360
💾 Saving index... ntotal=163,840
💾 Saving index... ntotal=184,320
✅ Progress: 200,704 vectors
💾 Saving index... ntotal=204,800
💾 Saving index... ntotal=225,280
💾 Saving index... ntotal=245,760
💾 Saving index... ntotal=266,240
💾 Saving index... ntotal=286,720
💾 Saving index... ntotal=307,200
💾 Saving index... ntotal=327,680
💾 Saving index... ntotal=348,160
💾 Saving index... ntotal=368,640
💾 Saving index... ntotal=389,120
✅ Progress: 401,408 vectors
💾 Saving index... ntotal=409,600
💾 Saving index... ntotal=430,080
💾 Saving index... ntotal=450,560
💾 Saving index... ntotal=471,040
💾 Saving index... ntotal=491,520
💾 Saving index... ntotal=512,000
💾 Saving index... ntotal=532,480
💾 Saving index... ntotal=552,960
💾 Saving in

In [3]:
import os, json, sqlite3
from tqdm import tqdm

META_PATH = "fever_dense/passage_meta.jsonl"
BM25_DB   = "fever_dense/bm25_passages.sqlite"

def build_bm25_fts(meta_path=META_PATH, bm25_db=BM25_DB, commit_every=50000):
    assert os.path.exists(meta_path), f"Missing meta: {meta_path}"

    if os.path.exists(bm25_db):
        print("✅ BM25 DB already exists:", bm25_db)
        return

    conn = sqlite3.connect(bm25_db)
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS passages (
        vector_id INTEGER PRIMARY KEY,
        title TEXT NOT NULL,
        start_line INTEGER NOT NULL,
        end_line INTEGER NOT NULL,
        text TEXT NOT NULL
    );
    """)
    conn.commit()

    # FTS5 virtual table
    cur.execute("""
    CREATE VIRTUAL TABLE IF NOT EXISTS passages_fts
    USING fts5(text, content='passages', content_rowid='vector_id');
    """)
    conn.commit()

    # Count lines for progress bar
    with open(meta_path, "r", encoding="utf-8") as f:
        total = sum(1 for _ in f)

    buf_passages, buf_fts = [], []
    inserted = 0

    with open(meta_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, total=total, desc="Building BM25 FTS5"):
            r = json.loads(line)
            vid = int(r["vector_id"])
            title = r["title"]
            a = int(r["start_line"])
            b = int(r["end_line"])
            text = r["text"]

            buf_passages.append((vid, title, a, b, text))
            buf_fts.append((vid, text))

            if len(buf_passages) >= commit_every:
                cur.executemany("INSERT OR REPLACE INTO passages VALUES (?, ?, ?, ?, ?);", buf_passages)
                cur.executemany("INSERT INTO passages_fts(rowid, text) VALUES (?, ?);", buf_fts)
                conn.commit()
                inserted += len(buf_passages)
                buf_passages, buf_fts = [], []

        if buf_passages:
            cur.executemany("INSERT OR REPLACE INTO passages VALUES (?, ?, ?, ?, ?);", buf_passages)
            cur.executemany("INSERT INTO passages_fts(rowid, text) VALUES (?, ?);", buf_fts)
            conn.commit()
            inserted += len(buf_passages)

    cur.execute("CREATE INDEX IF NOT EXISTS idx_passages_title ON passages(title);")
    conn.commit()
    conn.close()

    print("✅ BM25 DB built:", bm25_db)
    print("Inserted rows:", inserted)

build_bm25_fts()

✅ BM25 DB already exists: fever_dense/bm25_passages.sqlite


In [4]:
import os, re, json, sqlite3
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import torch

import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [5]:
# =========================
# PATHS
# =========================
DB_PATH     = "fever_data/fever_wiki.sqlite"           # your wiki sentences db (already built)
FEVER_DEV   = "fever_data/shared_task_dev.jsonl"       # labelled dev
FEVER_TRAIN = "fever_data/train.jsonl"                 # optional

OUT_DIR = "fever_dense"
os.makedirs(OUT_DIR, exist_ok=True)

INDEX_PATH = os.path.join(OUT_DIR, "faiss.index")
META_PATH  = os.path.join(OUT_DIR, "passage_meta.jsonl")
STATE_PATH = os.path.join(OUT_DIR, "build_state.json")

BM25_DB = os.path.join(OUT_DIR, "bm25_passages.sqlite")

# =========================
# MODELS
# =========================
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMB_DIM   = 384

VERIFIER_MODEL = "ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli"

# Reranker (optional but strong)
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE=="cuda" else None)

# =========================
# PASSAGE SETTINGS (Step 3)
# =========================
PASSAGE_SENTENCES = 3
MAX_SENT_CHARS    = 350
MAX_PASSAGE_CHARS = 1200
HNSW_M            = 32
BATCH_PASSAGES    = 4096
COMMIT_EVERY      = 50000  # save index periodically

# =========================
# EVAL SETTINGS
# =========================
TOPK_FINAL = 10
MAX_EVIDENCE_FOR_VERIFIER = 5
EVAL_SAMPLES = 10000  # set 19998 for full dev

# Candidates for hybrid/rerank
TOPK_DENSE_CAND = 50
TOPK_BM25_CAND  = 50

# Toggle these
RUN_BUILD_DENSE = False   # <-- set True only if you want to build FAISS again
RUN_BUILD_BM25  = False   # <-- set True only if BM25 DB not built

DEVICE: cuda | GPU: NVIDIA B200 MIG 1g.45gb


In [6]:
def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

def save_json(path: str, obj: dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: str) -> Optional[dict]:
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def clean_sentence(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    if "\t" in s:
        s = s.split("\t", 1)[0].strip()
    if len(s) > MAX_SENT_CHARS:
        s = s[:MAX_SENT_CHARS].rsplit(" ", 1)[0]
    return s

def normalize_embeddings(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / norms

# ---- Safe evidence parsing (fixes NoneType bug) ----
def parse_gold_evidence_sets_safe(evidence) -> List[List[Tuple[str, int]]]:
    if not evidence:
        return []
    if isinstance(evidence, str):
        evidence = json.loads(evidence)

    sets = []
    for ev_set in evidence:
        one = []
        for ev in ev_set:
            if (
                isinstance(ev, list)
                and len(ev) >= 4
                and ev[2] is not None
                and ev[3] is not None
            ):
                try:
                    one.append((str(ev[2]), int(ev[3])))
                except Exception:
                    continue
        if one:
            sets.append(one)
    return sets

def passage_covers_pair(meta_row: dict, pair: Tuple[str, int]) -> bool:
    t, line_id = pair
    return (
        meta_row["title"] == t and
        int(meta_row["start_line"]) <= int(line_id) <= int(meta_row["end_line"])
    )

def fever_score_one(gold_label: str, pred_label: str,
                    gold_sets: List[List[Tuple[str, int]]],
                    retrieved_rows: List[dict]) -> int:
    # classic FEVER score
    if pred_label != gold_label:
        return 0
    if gold_label == "NOT ENOUGH INFO":
        return 1
    # must retrieve at least one COMPLETE evidence set
    for gs in gold_sets:
        ok = True
        for pair in gs:
            if not any(passage_covers_pair(r, pair) for r in retrieved_rows):
                ok = False
                break
        if ok:
            return 1
    return 0

def load_meta(meta_path: str) -> Dict[int, dict]:
    meta = {}
    with open(meta_path, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            meta[int(r["vector_id"])] = r
    return meta

In [7]:
class WikiPassageStreamer:
    """
    Streams 3-sentence passages from wiki_sentences(title, line_id, sentence)
    in DB_PATH.
    """
    def __init__(self, db_path: str):
        self.conn = sqlite3.connect(db_path)
        self.cur = self.conn.cursor()

    def stream_passages(self, start_after: Optional[Tuple[str, int]] = None):
        if start_after is None:
            q = """
            SELECT title, line_id, sentence
            FROM wiki_sentences
            ORDER BY title ASC, line_id ASC
            """
            params = ()
        else:
            last_title, last_line = start_after
            q = """
            SELECT title, line_id, sentence
            FROM wiki_sentences
            WHERE (title > ?) OR (title = ? AND line_id > ?)
            ORDER BY title ASC, line_id ASC
            """
            params = (last_title, last_title, int(last_line))

        cur_title = None
        buf_lines: List[Tuple[int, str]] = []

        def emit_passages(title: str, lines: List[Tuple[int, str]]):
            emitted = []
            i = 0
            n = len(lines)
            while i + PASSAGE_SENTENCES <= n:
                chunk = lines[i:i+PASSAGE_SENTENCES]
                start_line = chunk[0][0]
                end_line   = chunk[-1][0]
                text = " ".join([c[1] for c in chunk]).strip()
                if len(text) > MAX_PASSAGE_CHARS:
                    text = text[:MAX_PASSAGE_CHARS].rsplit(" ", 1)[0]
                emitted.append({
                    "title": title,
                    "start_line": start_line,
                    "end_line": end_line,
                    "text": text
                })
                i += PASSAGE_SENTENCES
            return emitted, lines[i:]

        for title, line_id, sent in self.cur.execute(q, params):
            title = str(title)
            line_id = int(line_id)
            sent = clean_sentence(sent)
            if not sent:
                continue

            if cur_title is None:
                cur_title = title

            if title != cur_title:
                emitted, _ = emit_passages(cur_title, buf_lines)
                for p in emitted:
                    yield p
                cur_title = title
                buf_lines = []

            buf_lines.append((line_id, sent))

            if len(buf_lines) >= 6000:
                emitted, rem = emit_passages(cur_title, buf_lines)
                for p in emitted:
                    yield p
                buf_lines = rem

        if cur_title and buf_lines:
            emitted, _ = emit_passages(cur_title, buf_lines)
            for p in emitted:
                yield p

    def close(self):
        self.conn.close()

In [8]:
def load_or_create_index(path: str) -> faiss.Index:
    if os.path.exists(path):
        print("✅ Loading FAISS index:", path)
        return faiss.read_index(path)
    print("🧠 Creating FAISS HNSW (cosine via normalized vectors + IP)")
    idx = faiss.IndexHNSWFlat(EMB_DIM, HNSW_M, faiss.METRIC_INNER_PRODUCT)
    idx.hnsw.efConstruction = 200
    idx.hnsw.efSearch = 128
    return idx

def append_meta(meta_path: str, rows: List[dict]):
    with open(meta_path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def build_dense_index():
    assert os.path.exists(DB_PATH), f"Missing wiki db: {DB_PATH}"

    state = load_json(STATE_PATH) or {}
    resume_after = state.get("resume_after")
    resume_tuple = (resume_after["title"], int(resume_after["line_id"])) if resume_after else None
    if resume_tuple:
        print("🔁 Resuming after:", resume_tuple)

    idx = load_or_create_index(INDEX_PATH)

    print(f"🔧 Loading SBERT for indexing: {EMB_MODEL} on {DEVICE}")
    embedder = SentenceTransformer(EMB_MODEL, device=DEVICE)
    if DEVICE == "cuda":
        embedder = embedder.half()

    if not os.path.exists(META_PATH):
        open(META_PATH, "w", encoding="utf-8").close()

    streamer = WikiPassageStreamer(DB_PATH)
    buffer_passages = []
    last_seen = None
    added_since_save = 0

    print("📌 Current vectors in index:", idx.ntotal)
    print("🚀 Building dense passage index...")

    def flush():
        nonlocal buffer_passages, last_seen, added_since_save
        if not buffer_passages:
            return

        texts = [p["text"] for p in buffer_passages]
        emb = embedder.encode(
            texts,
            batch_size=128 if DEVICE == "cuda" else 32,
            convert_to_numpy=True,
            normalize_embeddings=False,
            show_progress_bar=False
        ).astype("float32")
        emb = normalize_embeddings(emb)

        before = idx.ntotal
        idx.add(emb)
        n_added = emb.shape[0]
        added_since_save += n_added

        meta_rows = []
        for i, p in enumerate(buffer_passages):
            vid = before + i
            meta_rows.append({
                "vector_id": int(vid),
                "title": p["title"],
                "start_line": int(p["start_line"]),
                "end_line": int(p["end_line"]),
                "text": p["text"]
            })
        append_meta(META_PATH, meta_rows)

        if last_seen:
            save_json(STATE_PATH, {"resume_after": {"title": last_seen[0], "line_id": int(last_seen[1])}})

        if added_since_save >= COMMIT_EVERY:
            print(f"💾 Saving index... ntotal={idx.ntotal}")
            faiss.write_index(idx, INDEX_PATH)
            added_since_save = 0

        buffer_passages = []

    for p in streamer.stream_passages(start_after=resume_tuple):
        buffer_passages.append(p)
        last_seen = (p["title"], p["end_line"])
        if len(buffer_passages) >= BATCH_PASSAGES:
            flush()

    flush()
    print("💾 Final save index...")
    faiss.write_index(idx, INDEX_PATH)
    if last_seen:
        save_json(STATE_PATH, {"resume_after": {"title": last_seen[0], "line_id": int(last_seen[1])}})

    streamer.close()

    print("✅ Done building index.")
    print("Total vectors:", idx.ntotal)
    print("Meta:", META_PATH)
    print("Index:", INDEX_PATH)

if RUN_BUILD_DENSE:
    build_dense_index()
else:
    print("⏭️ Skipping dense build (RUN_BUILD_DENSE=False)")
    print("Expect existing:", INDEX_PATH, META_PATH)

⏭️ Skipping dense build (RUN_BUILD_DENSE=False)
Expect existing: fever_dense/faiss.index fever_dense/passage_meta.jsonl


In [9]:
class NliVerifier:
    """
    entail -> SUPPORTS
    contrad -> REFUTES
    neutral -> NEI
    aggregation:
      if any entail -> SUPPORTS
      elif any contradiction -> REFUTES
      else -> NEI
    """
    def __init__(self, model_name: str, device: str):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
        self.model.eval()
        self.device = device
        self.id2nli = self.model.config.id2label

    @torch.no_grad()
    def nli_probs(self, premise: str, hypothesis: str):
        inp = self.tokenizer(premise, hypothesis, truncation=True, max_length=512, return_tensors="pt").to(self.device)
        logits = self.model(**inp).logits[0]
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()

        names = [self.id2nli[i].lower() for i in range(len(probs))]

        def find_idx(key: str):
            for i, n in enumerate(names):
                if key in n:
                    return i
            return None

        idx_con = find_idx("contrad")
        idx_neu = find_idx("neutral")
        idx_ent = find_idx("entail")
        if idx_con is None or idx_neu is None or idx_ent is None:
            idx_con, idx_neu, idx_ent = 0, 1, 2

        return float(probs[idx_con]), float(probs[idx_neu]), float(probs[idx_ent])

    def verify_claim(self, claim: str, passages: List[str]) -> str:
        any_ent = False
        any_con = False

        for p in passages:
            p_con, p_neu, p_ent = self.nli_probs(p, claim)
            if p_ent >= max(p_con, p_neu):
                any_ent = True
            elif p_con >= max(p_ent, p_neu):
                any_con = True

        if any_ent:
            return "SUPPORTS"
        if any_con:
            return "REFUTES"
        return "NOT ENOUGH INFO"


# --------- BM25 Query (robust + entity-ish) ----------
STOPWORDS = {
    "a","an","the","is","are","was","were","be","been","being","of","in","on","at","to","for","and","or","by","with",
    "from","as","that","this","it","its","into","than","then","but","not","no","yes","do","does","did","done","has",
    "have","had","he","she","they","them","his","her","their","we","you","i","my","your","our","also"
}

def fts5_query_smart(claim: str) -> str:
    text = (claim or "").lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    terms = [t for t in text.split() if t not in STOPWORDS and len(t) >= 3]
    if not terms:
        return ""

    terms = sorted(terms, key=len, reverse=True)
    core = terms[:5]
    core_q = " AND ".join(core)
    extra = terms[5:20]
    if extra:
        return f"({core_q}) OR ({' OR '.join(extra)})"
    return core_q

def bm25_search(conn, claim: str, topk: int) -> List[int]:
    q = fts5_query_smart(claim)
    if not q:
        return []
    cur = conn.cursor()
    rows = cur.execute(
        "SELECT rowid, bm25(passages_fts) AS score "
        "FROM passages_fts WHERE passages_fts MATCH ? "
        "ORDER BY score ASC LIMIT ?;",
        (q, topk)
    ).fetchall()
    return [int(r[0]) for r in rows]

In [10]:
def evaluate_dense():
    assert os.path.exists(INDEX_PATH), "Missing FAISS index"
    assert os.path.exists(META_PATH), "Missing meta jsonl"
    assert os.path.exists(FEVER_DEV), "Missing FEVER dev jsonl"

    print("✅ Loading FAISS...")
    idx = faiss.read_index(INDEX_PATH)
    print("FAISS ntotal:", idx.ntotal)

    print("✅ Loading meta into RAM...")
    meta = load_meta(META_PATH)
    print("Meta rows:", len(meta))

    print("✅ Loading SBERT...")
    embedder = SentenceTransformer(EMB_MODEL, device=DEVICE)
    if DEVICE == "cuda":
        embedder = embedder.half()

    print("✅ Loading verifier...")
    verifier = NliVerifier(VERIFIER_MODEL, DEVICE)

    # metrics
    n_retr = 0
    recall_hits = 0
    mrr_sum = 0.0
    n_label = 0
    label_correct = 0
    fever_total = 0
    fever_hits = 0

    shown = 0

    for ex in iter_jsonl(FEVER_DEV):
        claim = ex.get("claim", "")
        gold  = ex.get("label", "")

        q = embedder.encode([claim], convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False).astype("float32")
        q = normalize_embeddings(q)

        D, I = idx.search(q, TOPK_FINAL)
        vids = [int(v) for v in I[0].tolist() if int(v) >= 0]
        retrieved = [meta[v] for v in vids if v in meta]

        gold_sets = parse_gold_evidence_sets_safe(ex.get("evidence"))

        # retrieval metrics (supports/refutes only): ANY gold line hit
        if gold in ("SUPPORTS", "REFUTES") and gold_sets:
            n_retr += 1
            all_gold_pairs = [pair for s in gold_sets for pair in s]
            found_rank = None
            for r, row in enumerate(retrieved, start=1):
                if any(passage_covers_pair(row, pair) for pair in all_gold_pairs):
                    found_rank = r
                    break
            if found_rank is not None:
                recall_hits += 1
                mrr_sum += 1.0 / found_rank

        # verify
        top_passages = [r["text"] for r in retrieved[:MAX_EVIDENCE_FOR_VERIFIER]]
        pred = verifier.verify_claim(claim, top_passages)

        n_label += 1
        if pred == gold:
            label_correct += 1

        fever_total += 1
        fever_hits += fever_score_one(gold, pred, gold_sets, retrieved)

        if shown < 2:
            shown += 1
            print("\n===== SAMPLE (DENSE) =====")
            print("CLAIM:", claim)
            print("GOLD :", gold)
            print("PRED :", pred)

        if n_label >= EVAL_SAMPLES:
            break

    recall = recall_hits / n_retr if n_retr else 0.0
    mrr    = mrr_sum / n_retr if n_retr else 0.0
    acc    = label_correct / n_label if n_label else 0.0
    fever  = fever_hits / fever_total if fever_total else 0.0

    print("\n================ RESULTS (DENSE) ================")
    print(f"Eval samples (label)  : {n_label}")
    print(f"Label Accuracy        : {acc:.4f}")
    print()
    print(f"Eval samples (retr supports/refutes): {n_retr}")
    print(f"Recall@{TOPK_FINAL}          : {recall:.4f}")
    print(f"MRR@{TOPK_FINAL}             : {mrr:.4f}")
    print()
    print(f"FEVER Score           : {fever:.4f}")

evaluate_dense()

✅ Loading FAISS...
FAISS ntotal: 6473073
✅ Loading meta into RAM...
Meta rows: 6473073
✅ Loading SBERT...
✅ Loading verifier...


Some weights of the model checkpoint at ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



===== SAMPLE (DENSE) =====
CLAIM: Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.
GOLD : NOT ENOUGH INFO
PRED : SUPPORTS

===== SAMPLE (DENSE) =====
CLAIM: Tilda Swinton is a vegan.
GOLD : NOT ENOUGH INFO
PRED : SUPPORTS

================ RESULTS (DENSE) ================
Eval samples (label)  : 10000
Label Accuracy        : 0.5788

Eval samples (retr supports/refutes): 6666
Recall@10          : 0.4947
MRR@10             : 0.3085

FEVER Score           : 0.3296


In [14]:
def evaluate_bm25_only():
    """
    BM25-only retrieval baseline:
      - retrieve TOPK_FINAL passages using bm25_search()
      - verify using the same NLI verifier on top MAX_EVIDENCE_FOR_VERIFIER passages
      - compute label accuracy, Recall@TOPK_FINAL, MRR@TOPK_FINAL, FEVER score

    Assumes the following are already defined in your notebook/script:
      INDEX_PATH, META_PATH, BM25_DB, FEVER_DEV
      TOPK_FINAL, TOPK_BM25_CAND, MAX_EVIDENCE_FOR_VERIFIER, EVAL_SAMPLES, DEVICE
      VERIFIER_MODEL
      iter_jsonl(), load_meta(), bm25_search(), parse_gold_evidence_sets_safe()
      passage_covers_pair(), fever_score_one()
      NliVerifier
    """
    import os, sqlite3

    assert os.path.exists(META_PATH), "Missing meta jsonl"
    assert os.path.exists(BM25_DB), "Missing BM25 sqlite db"
    assert os.path.exists(FEVER_DEV), "Missing FEVER dev jsonl"

    print("✅ Loading meta into RAM...")
    meta = load_meta(META_PATH)
    print("Meta rows:", len(meta))

    print("✅ Opening BM25 DB...")
    conn = sqlite3.connect(BM25_DB)

    print("✅ Loading verifier...")
    verifier = NliVerifier(VERIFIER_MODEL, DEVICE)

    # metrics
    n_retr = 0
    recall_hits = 0
    mrr_sum = 0.0
    n_label = 0
    label_correct = 0
    fever_total = 0
    fever_hits = 0

    shown = 0

    for ex in iter_jsonl(FEVER_DEV):
        claim = ex.get("claim", "")
        gold  = ex.get("label", "")

        # BM25 retrieve candidate ids
        bm25_ids = bm25_search(conn, claim, TOPK_BM25_CAND)

        # Keep order, dedup, cut to TOPK_FINAL
        seen = set()
        final_ids = []
        for vid in bm25_ids:
            vid = int(vid)
            if vid not in seen:
                seen.add(vid)
                final_ids.append(vid)
            if len(final_ids) >= TOPK_FINAL:
                break

        retrieved = [meta[v] for v in final_ids if v in meta]
        gold_sets = parse_gold_evidence_sets_safe(ex.get("evidence"))

        # retrieval metrics only for supports/refutes and when evidence exists
        if gold in ("SUPPORTS", "REFUTES") and gold_sets:
            n_retr += 1
            all_gold_pairs = [pair for s in gold_sets for pair in s]
            found_rank = None
            for r, row in enumerate(retrieved, start=1):
                if any(passage_covers_pair(row, pair) for pair in all_gold_pairs):
                    found_rank = r
                    break
            if found_rank is not None:
                recall_hits += 1
                mrr_sum += 1.0 / found_rank

        # verify (same verifier logic)
        top_passages = [r["text"] for r in retrieved[:MAX_EVIDENCE_FOR_VERIFIER]]
        pred = verifier.verify_claim(claim, top_passages)

        n_label += 1
        if pred == gold:
            label_correct += 1

        fever_total += 1
        fever_hits += fever_score_one(gold, pred, gold_sets, retrieved)

        if shown < 2:
            shown += 1
            print("\n===== SAMPLE (BM25) =====")
            print("CLAIM:", claim)
            print("GOLD :", gold)
            print("PRED :", pred)

        if n_label >= EVAL_SAMPLES:
            break

    conn.close()

    recall = recall_hits / n_retr if n_retr else 0.0
    mrr    = mrr_sum / n_retr if n_retr else 0.0
    acc    = label_correct / n_label if n_label else 0.0
    fever  = fever_hits / fever_total if fever_total else 0.0

    print("\n================ RESULTS (BM25 ONLY) ================")
    print(f"Eval samples (label)  : {n_label}")
    print(f"Label Accuracy        : {acc:.4f}")
    print()
    print(f"Eval samples (retr supports/refutes): {n_retr}")
    print(f"Recall@{TOPK_FINAL}          : {recall:.4f}")
    print(f"MRR@{TOPK_FINAL}             : {mrr:.4f}")
    print()
    print(f"FEVER Score           : {fever:.4f}")


# run
evaluate_bm25_only()

✅ Loading meta into RAM...
Meta rows: 6473073
✅ Opening BM25 DB...
✅ Loading verifier...


Some weights of the model checkpoint at ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



===== SAMPLE (BM25) =====
CLAIM: Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League.
GOLD : NOT ENOUGH INFO
PRED : NOT ENOUGH INFO

===== SAMPLE (BM25) =====
CLAIM: Tilda Swinton is a vegan.
GOLD : NOT ENOUGH INFO
PRED : NOT ENOUGH INFO

================ RESULTS (BM25 ONLY) ================
Eval samples (label)  : 10000
Label Accuracy        : 0.5080

Eval samples (retr supports/refutes): 6666
Recall@10          : 0.2132
MRR@10             : 0.1528

FEVER Score           : 0.3435


In [15]:
def evaluate_hybrid_union():
    assert os.path.exists(INDEX_PATH)
    assert os.path.exists(META_PATH)
    assert os.path.exists(BM25_DB)

    print("✅ Loading FAISS...")
    idx = faiss.read_index(INDEX_PATH)

    print("✅ Loading meta into RAM...")
    meta = load_meta(META_PATH)

    print("✅ Opening BM25 DB...")
    conn = sqlite3.connect(BM25_DB)

    print("✅ Loading SBERT...")
    embedder = SentenceTransformer(EMB_MODEL, device=DEVICE)
    if DEVICE == "cuda":
        embedder = embedder.half()

    print("✅ Loading verifier...")
    verifier = NliVerifier(VERIFIER_MODEL, DEVICE)

    n_retr = 0
    recall_hits = 0
    mrr_sum = 0.0
    n_label = 0
    label_correct = 0
    fever_total = 0
    fever_hits = 0

    for ex in iter_jsonl(FEVER_DEV):
        claim = ex.get("claim", "")
        gold  = ex.get("label", "")

        # dense candidates
        q = embedder.encode([claim], convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False).astype("float32")
        q = normalize_embeddings(q)
        D, I = idx.search(q, TOPK_DENSE_CAND)
        dense_ids = [int(v) for v in I[0].tolist() if int(v) >= 0]

        # bm25 candidates
        bm25_ids = bm25_search(conn, claim, TOPK_BM25_CAND)

        # union, keep dense order first then bm25
        seen = set()
        final_ids = []
        for vid in dense_ids + bm25_ids:
            if vid not in seen:
                seen.add(vid)
                final_ids.append(vid)
            if len(final_ids) >= TOPK_FINAL:
                break

        retrieved = [meta[v] for v in final_ids if v in meta]
        gold_sets = parse_gold_evidence_sets_safe(ex.get("evidence"))

        if gold in ("SUPPORTS", "REFUTES") and gold_sets:
            n_retr += 1
            all_gold_pairs = [pair for s in gold_sets for pair in s]
            found_rank = None
            for r, row in enumerate(retrieved, start=1):
                if any(passage_covers_pair(row, pair) for pair in all_gold_pairs):
                    found_rank = r
                    break
            if found_rank is not None:
                recall_hits += 1
                mrr_sum += 1.0 / found_rank

        top_passages = [r["text"] for r in retrieved[:MAX_EVIDENCE_FOR_VERIFIER]]
        pred = verifier.verify_claim(claim, top_passages)

        n_label += 1
        if pred == gold:
            label_correct += 1

        fever_total += 1
        fever_hits += fever_score_one(gold, pred, gold_sets, retrieved)

        if n_label >= EVAL_SAMPLES:
            break

    conn.close()

    recall = recall_hits / n_retr if n_retr else 0.0
    mrr    = mrr_sum / n_retr if n_retr else 0.0
    acc    = label_correct / n_label if n_label else 0.0
    fever  = fever_hits / fever_total if fever_total else 0.0

    print("\n================ RESULTS (HYBRID UNION) ================")
    print(f"Eval samples (label)  : {n_label}")
    print(f"Label Accuracy        : {acc:.4f}")
    print()
    print(f"Eval samples (retr supports/refutes): {n_retr}")
    print(f"Recall@{TOPK_FINAL}          : {recall:.4f}")
    print(f"MRR@{TOPK_FINAL}             : {mrr:.4f}")
    print()
    print(f"FEVER Score           : {fever:.4f}")

evaluate_hybrid_union()

✅ Loading FAISS...
✅ Loading meta into RAM...
✅ Opening BM25 DB...
✅ Loading SBERT...
✅ Loading verifier...


Some weights of the model checkpoint at ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



================ RESULTS (HYBRID UNION) ================
Eval samples (label)  : 10000
Label Accuracy        : 0.5788

Eval samples (retr supports/refutes): 6666
Recall@10          : 0.4947
MRR@10             : 0.3085

FEVER Score           : 0.3296


In [16]:
def evaluate_hybrid_rerank():
    assert os.path.exists(INDEX_PATH)
    assert os.path.exists(META_PATH)
    assert os.path.exists(BM25_DB)

    print("✅ Loading FAISS...")
    idx = faiss.read_index(INDEX_PATH)

    print("✅ Loading meta into RAM...")
    meta = load_meta(META_PATH)

    print("✅ Opening BM25 DB...")
    conn = sqlite3.connect(BM25_DB)

    print("✅ Loading SBERT...")
    embedder = SentenceTransformer(EMB_MODEL, device=DEVICE)
    if DEVICE == "cuda":
        embedder = embedder.half()

    print("✅ Loading reranker...")
    reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)

    print("✅ Loading verifier...")
    verifier = NliVerifier(VERIFIER_MODEL, DEVICE)

    def rerank(claim: str, candidates: List[dict], topk: int) -> List[dict]:
        if not candidates:
            return []
        pairs = [(claim, r["text"]) for r in candidates]
        scores = reranker.predict(pairs, batch_size=64)
        order = np.argsort(-scores)[:topk]
        return [candidates[i] for i in order]

    n_retr = 0
    recall_hits = 0
    mrr_sum = 0.0
    n_label = 0
    label_correct = 0
    fever_total = 0
    fever_hits = 0

    for ex in iter_jsonl(FEVER_DEV):
        claim = ex.get("claim", "")
        gold  = ex.get("label", "")

        # dense candidates
        q = embedder.encode([claim], convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False).astype("float32")
        q = normalize_embeddings(q)
        D, I = idx.search(q, TOPK_DENSE_CAND)
        dense_ids = [int(v) for v in I[0].tolist() if int(v) >= 0]

        # bm25 candidates
        bm25_ids = bm25_search(conn, claim, TOPK_BM25_CAND)

        # union candidates (up to 100)
        seen = set()
        cand_ids = []
        for vid in dense_ids + bm25_ids:
            if vid not in seen:
                seen.add(vid)
                cand_ids.append(vid)

        candidates = [meta[v] for v in cand_ids if v in meta]

        # rerank to top-10
        retrieved = rerank(claim, candidates, TOPK_FINAL)

        gold_sets = parse_gold_evidence_sets_safe(ex.get("evidence"))

        if gold in ("SUPPORTS", "REFUTES") and gold_sets:
            n_retr += 1
            all_gold_pairs = [pair for s in gold_sets for pair in s]
            found_rank = None
            for r, row in enumerate(retrieved, start=1):
                if any(passage_covers_pair(row, pair) for pair in all_gold_pairs):
                    found_rank = r
                    break
            if found_rank is not None:
                recall_hits += 1
                mrr_sum += 1.0 / found_rank

        top_passages = [r["text"] for r in retrieved[:MAX_EVIDENCE_FOR_VERIFIER]]
        pred = verifier.verify_claim(claim, top_passages)

        n_label += 1
        if pred == gold:
            label_correct += 1

        fever_total += 1
        fever_hits += fever_score_one(gold, pred, gold_sets, retrieved)

        if n_label >= EVAL_SAMPLES:
            break

    conn.close()

    recall = recall_hits / n_retr if n_retr else 0.0
    mrr    = mrr_sum / n_retr if n_retr else 0.0
    acc    = label_correct / n_label if n_label else 0.0
    fever  = fever_hits / fever_total if fever_total else 0.0

    print("\n================ RESULTS (HYBRID + RERANK) ================")
    print(f"Eval samples (label)  : {n_label}")
    print(f"Label Accuracy        : {acc:.4f}")
    print()
    print(f"Eval samples (retr supports/refutes): {n_retr}")
    print(f"Recall@{TOPK_FINAL}          : {recall:.4f}")
    print(f"MRR@{TOPK_FINAL}             : {mrr:.4f}")
    print()
    print(f"FEVER Score           : {fever:.4f}")

evaluate_hybrid_rerank()

✅ Loading FAISS...
✅ Loading meta into RAM...
✅ Opening BM25 DB...
✅ Loading SBERT...
✅ Loading reranker...
✅ Loading verifier...


Some weights of the model checkpoint at ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



================ RESULTS (HYBRID + RERANK) ================
Eval samples (label)  : 10000
Label Accuracy        : 0.5874

Eval samples (retr supports/refutes): 6666
Recall@10          : 0.6412
MRR@10             : 0.5409

FEVER Score           : 0.3935


In [15]:
def fact_gate_pass_drop(article: str,
                        idx, meta, embedder, verifier,
                        reranker=None,
                        max_claims: int = 6,
                        topk_dense: int = 50,
                        topk_final: int = 10,
                        max_evidence: int = 5):

    claims = extract_claims(article, max_claims=max_claims)

    n_support = 0
    n_refute  = 0
    n_nei     = 0

    detailed = []

    for c in claims:
        rows = retrieve_evidence(c, idx, meta, embedder, reranker,
                                 topk_dense=topk_dense,
                                 topk_final=topk_final)

        label, scores, chosen = verify_with_scores(
            verifier, c, rows, max_evidence=max_evidence
        )

        if label == "SUPPORTS":
            n_support += 1
        elif label == "REFUTES":
            n_refute += 1
        else:
            n_nei += 1

        detailed.append({
            "claim": c,
            "label": label,
            "scores": scores
        })

    total = max(1, len(claims))

    # ===== FINAL DECISION RULE =====
    if n_refute > 0:
        decision = "DROP"
    elif n_support == 0:
        # nothing verifiable as true
        decision = "DROP"
    else:
        decision = "PASS"

    return {
        "decision": decision,
        "counts": {
            "supports": n_support,
            "refutes": n_refute,
            "nei": n_nei
        },
        "claims": detailed
    }

In [16]:
from sentence_transformers import CrossEncoder
import json, sqlite3
import numpy as np

# -----------------------------
# PASS/DROP mapping (prototype)
# -----------------------------
def label_to_passdrop(label: str) -> str:
    return "PASS" if label == "SUPPORTS" else "DROP"


# -----------------------------
# Hybrid retrieval + rerank (reuse your exact logic)
# -----------------------------
def retrieve_hybrid_rerank(claim: str, idx, meta, conn, embedder, reranker,
                          topk_dense=TOPK_DENSE_CAND,
                          topk_bm25=TOPK_BM25_CAND,
                          topk_final=TOPK_FINAL):
    # dense candidates
    q = embedder.encode([claim], convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False).astype("float32")
    q = normalize_embeddings(q)
    D, I = idx.search(q, topk_dense)
    dense_ids = [int(v) for v in I[0].tolist() if int(v) >= 0]

    # bm25 candidates
    bm25_ids = bm25_search(conn, claim, topk_bm25)

    # union candidates
    seen = set()
    cand_ids = []
    for vid in dense_ids + bm25_ids:
        if vid not in seen:
            seen.add(vid)
            cand_ids.append(vid)

    candidates = [meta[v] for v in cand_ids if v in meta]

    # rerank
    if not candidates:
        return []

    pairs = [(claim, r["text"]) for r in candidates]
    scores = reranker.predict(pairs, batch_size=64)
    order = np.argsort(-scores)[:topk_final]
    retrieved = [candidates[i] for i in order]

    return retrieved


# -----------------------------
# FACT GATE runner on FEVER dev (2000 samples)
# -----------------------------
def run_fact_gate_on_fever_dev(
    out_path="outputs/factgate_fever_dev_2000.jsonl",
    n_samples=EVAL_SAMPLES
):
    os.makedirs("outputs", exist_ok=True)

    assert os.path.exists(INDEX_PATH)
    assert os.path.exists(META_PATH)
    assert os.path.exists(BM25_DB)

    print("✅ Loading FAISS...")
    idx = faiss.read_index(INDEX_PATH)

    print("✅ Loading meta...")
    meta = load_meta(META_PATH)

    print("✅ Opening BM25 DB...")
    conn = sqlite3.connect(BM25_DB)

    print("✅ Loading SBERT...")
    embedder = SentenceTransformer(EMB_MODEL, device=DEVICE)
    if DEVICE == "cuda":
        embedder = embedder.half()

    print("✅ Loading reranker...")
    reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)

    print("✅ Loading verifier...")
    verifier = NliVerifier(VERIFIER_MODEL, DEVICE)

    # stats
    total = 0
    pass_cnt = 0
    drop_cnt = 0

    # optional: gate accuracy against your prototype truth:
    # gold SUPPORTS => PASS else DROP
    gate_correct = 0

    with open(out_path, "w", encoding="utf-8") as wf:
        for ex in iter_jsonl(FEVER_DEV):
            claim = ex.get("claim", "")
            gold_label = ex.get("label", "")

            retrieved = retrieve_hybrid_rerank(
                claim, idx, meta, conn, embedder, reranker,
                topk_dense=TOPK_DENSE_CAND,
                topk_bm25=TOPK_BM25_CAND,
                topk_final=TOPK_FINAL
            )

            top_passages = [r["text"] for r in retrieved[:MAX_EVIDENCE_FOR_VERIFIER]]
            pred_label = verifier.verify_claim(claim, top_passages)

            decision = label_to_passdrop(pred_label)
            gold_gate = label_to_passdrop(gold_label)  # your prototype mapping

            if decision == "PASS":
                pass_cnt += 1
            else:
                drop_cnt += 1

            if decision == gold_gate:
                gate_correct += 1

            row = {
                "id": ex.get("id", None),
                "claim": claim,
                "gold_label": gold_label,
                "pred_label": pred_label,
                "decision": decision,
                "gold_gate": gold_gate,
                "retrieved": [
                    {
                        "title": r["title"],
                        "start_line": int(r["start_line"]),
                        "end_line": int(r["end_line"]),
                        "text": r["text"][:350]
                    }
                    for r in retrieved[:TOPK_FINAL]
                ]
            }

            wf.write(json.dumps(row, ensure_ascii=False) + "\n")

            total += 1
            if total >= n_samples:
                break

    conn.close()

    print("\n================ FACT GATE SUMMARY ================")
    print("Processed samples:", total)
    print("PASS:", pass_cnt, f"({pass_cnt/total:.3f})")
    print("DROP:", drop_cnt, f"({drop_cnt/total:.3f})")
    print("Gate Accuracy vs prototype mapping:", f"{gate_correct/total:.4f}")
    print("Saved JSONL:", out_path)


run_fact_gate_on_fever_dev()

✅ Loading FAISS...
✅ Loading meta...
✅ Opening BM25 DB...
✅ Loading SBERT...
✅ Loading reranker...
✅ Loading verifier...


Some weights of the model checkpoint at ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



================ FACT GATE SUMMARY ================
Processed samples: 10000
PASS: 5727 (0.573)
DROP: 4273 (0.427)
Gate Accuracy vs prototype mapping: 0.7000
Saved JSONL: outputs/factgate_fever_dev_2000.jsonl


In [17]:
import json

PASS_FILE = "outputs/factgate_fever_dev_2000.jsonl"
pass_rows = []
with open(PASS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["decision"] == "PASS":
            pass_rows.append(r)

print("PASS rows:", len(pass_rows))

PASS rows: 5727


In [18]:
!pip -q install -U fsspec huggingface_hub datasets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.26a0+c5e1555-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.14.0-py3.12.egg is depreca

In [19]:
from huggingface_hub import snapshot_download
from datasets import load_dataset
import os

local_dir = snapshot_download(
    repo_id="horychtom/BASIL",
    repo_type="dataset",
    local_dir="basil_repo",
    local_dir_use_symlinks=False
)
print("✅ Downloaded to:", local_dir)

parquet_path = os.path.join(local_dir, "data", "basil.parquet")
print("✅ Parquet path:", parquet_path, "exists?", os.path.exists(parquet_path))

ds_raw = load_dataset("parquet", data_files=parquet_path)  # creates split "train"
print(ds_raw)
print("Columns:", ds_raw["train"].column_names)
print("Example:", ds_raw["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 18504.28it/s]


✅ Downloaded to: /workspace/basil_repo
✅ Parquet path: /workspace/basil_repo/data/basil.parquet exists? True


[WARNING  | datasets.builder   ]: Using custom data configuration default-59b5a0749af325d9
[WARNING  | datasets.builder   ]: Reusing dataset parquet (/root/.cache/huggingface/datasets/parquet/default-59b5a0749af325d9/0.0.0/0b6d5799bb726b24ad7fc7be720c170d8e497f575d02d47537de9a5bac074901)
100%|██████████| 1/1 [00:00<00:00, 504.30it/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'original_label', 'lex_label', 'info_label'],
        num_rows: 8018
    })
})
Columns: ['text', 'original_label', 'lex_label', 'info_label']
Example: {'text': 'He was also one of the wealthiest presidents ever to enter the White House, with far-reaching business connections that have already raised questions about where his interests end and the country’s begin.', 'original_label': 'info', 'lex_label': 0, 'info_label': 1}


In [20]:
from datasets import DatasetDict

full = ds_raw["train"]

# 80/10/10 split
tmp = full.train_test_split(test_size=0.2, seed=42)
tmp2 = tmp["test"].train_test_split(test_size=0.5, seed=42)

ds = DatasetDict({
    "train": tmp["train"],
    "validation": tmp2["train"],
    "test": tmp2["test"],
})

print({k: len(v) for k, v in ds.items()})

[WARNING  | datasets.arrow_dataset]: Loading cached split indices for dataset at /root/.cache/huggingface/datasets/parquet/default-59b5a0749af325d9/0.0.0/0b6d5799bb726b24ad7fc7be720c170d8e497f575d02d47537de9a5bac074901/cache-6d0007cda842a900.arrow and /root/.cache/huggingface/datasets/parquet/default-59b5a0749af325d9/0.0.0/0b6d5799bb726b24ad7fc7be720c170d8e497f575d02d47537de9a5bac074901/cache-bba54b86eb207be7.arrow
[WARNING  | datasets.arrow_dataset]: Loading cached split indices for dataset at /root/.cache/huggingface/datasets/parquet/default-59b5a0749af325d9/0.0.0/0b6d5799bb726b24ad7fc7be720c170d8e497f575d02d47537de9a5bac074901/cache-42eefc81e8f044a1.arrow and /root/.cache/huggingface/datasets/parquet/default-59b5a0749af325d9/0.0.0/0b6d5799bb726b24ad7fc7be720c170d8e497f575d02d47537de9a5bac074901/cache-702c6cb8eaf20439.arrow


{'train': 6414, 'validation': 802, 'test': 802}


In [21]:
import numpy as np

# We already created ds with train/validation/test splits earlier
# If not, use ds_raw["train"] and split first

def map_basil(ex):
    text = ex["text"]

    # Binary bias:
    # 1 = biased (lex OR info)
    # 0 = neutral
    y = 1 if ex["original_label"] in ["lex", "info"] else 0

    return {"text": text, "labels": y}

# Apply mapping
ds_m = ds.map(map_basil)

# Remove original columns to keep dataset clean
for split in ds_m.keys():
    drop_cols = [c for c in ds_m[split].column_names if c not in ["text", "labels"]]
    ds_m[split] = ds_m[split].remove_columns(drop_cols)

# ---- Sanity checks ----
print("✅ Sizes after mapping:", {k: len(v) for k, v in ds_m.items()})

ys = [ds_m["train"][i]["labels"] for i in range(min(5000, len(ds_m["train"])))]
print("Class counts (train sample):", np.bincount(ys))

print("Example mapped:", ds_m["train"][0])

[WARNING  | datasets.fingerprint]: Parameter 'function'=<function map_basil at 0x7158673f5b20> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.
100%|██████████| 802/802 [00:00<00:00, 19362.30ex/s]


✅ Sizes after mapping: {'train': 6414, 'validation': 802, 'test': 802}
Class counts (train sample): [3926 1074]
Example mapped: {'text': 'But an examination of his remarks at the ceremonies shows that he does not volunteer that his service never took him overseas.', 'labels': 0}


In [22]:
!pip -q install -U transformers accelerate evaluate scikit-learn

import os, json, random
import numpy as np
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, balanced_accuracy_score
)
from collections import Counter

# -------------------------
# Reproducibility
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -------------------------
# Config
# -------------------------
MODEL_NAME = "microsoft/deberta-v3-base"
SAVE_DIR = "bias_gate_deberta"
MAX_LEN = 256

id2label = {0: "NOT_BIASED", 1: "BIASED"}
label2id = {"NOT_BIASED": 0, "BIASED": 1}

splits = list(ds_m.keys())
TRAIN_SPLIT = "train"
VAL_SPLIT = "validation" if "validation" in splits else "dev"
TEST_SPLIT = "test" if "test" in splits else None

print("Using splits:", TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT)

# -------------------------
# Tokenization
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tok_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tok_ds = ds_m.map(tok_fn, batched=True)

keep_cols = ["input_ids", "attention_mask", "labels"]
for sp in tok_ds.keys():
    drop_cols = [c for c in tok_ds[sp].column_names if c not in keep_cols]
    tok_ds[sp] = tok_ds[sp].remove_columns(drop_cols)

# -------------------------
# Class weights
# -------------------------
train_labels = [int(tok_ds[TRAIN_SPLIT][i]["labels"]) for i in range(len(tok_ds[TRAIN_SPLIT]))]
counts = Counter(train_labels)
print("Train label counts:", counts)

c0 = counts.get(0, 1)
c1 = counts.get(1, 1)

w0 = 1.0 / c0
w1 = 1.0 / c1
weights = torch.tensor([w0, w1], dtype=torch.float32)
weights = weights / weights.mean()

print("Class weights:", weights.tolist())

# -------------------------
# Model
# -------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# -------------------------
# Custom Trainer
# -------------------------
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"].long()
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(weight=weights.to(logits.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")

    p, r, f1, sup = precision_recall_fscore_support(
        labels, preds, average=None, labels=[0,1], zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

    return {
        "acc": acc,
        "balanced_acc": bal_acc,
        "macro_f1": macro_f1,
        "f1_biased": float(f1[1]),
        "prec_biased": float(p[1]),
        "rec_biased": float(r[1]),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }

# -------------------------
# TrainingArguments
# -------------------------
args = TrainingArguments(
    output_dir="bias_gate_deberta_ckpts",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    num_train_epochs=3,
    warmup_ratio=0.06,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    max_grad_norm=1.0,   # gradient clipping
    fp16=torch.cuda.is_available(),
    report_to=[],
    save_total_limit=2,
    seed=SEED
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=tok_ds[TRAIN_SPLIT],
    eval_dataset=tok_ds[VAL_SPLIT],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()

# Save model
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("✅ Saved model to:", SAVE_DIR)

# -------------------------
# Threshold Optimization
# -------------------------
print("\n🔧 Threshold tuning...")

pred = trainer.predict(tok_ds[VAL_SPLIT])
logits = pred.predictions
labels = pred.label_ids

probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
p_biased = probs[:, 1]

best_thr = 0.5
best_f1 = 0

for thr in np.linspace(0.1, 0.9, 33):
    yhat = (p_biased >= thr).astype(int)
    f1b = f1_score(labels, yhat, pos_label=1)
    if f1b > best_f1:
        best_f1 = f1b
        best_thr = float(thr)

print("Best threshold:", best_thr, "F1_biased:", round(best_f1,4))

with open(os.path.join(SAVE_DIR, "gate_threshold.json"), "w") as f:
    json.dump({"threshold": best_thr, "val_f1_biased": best_f1}, f, indent=2)

print("Saved threshold.")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.26a0+c5e1555-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.14.0-py3.12.egg is depreca

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
[WARNING  | datasets.arrow_dataset]: Loading cached processed dataset at /root/.cache/huggingface/datasets/parquet/default-59b5a0749af325d9/0.0.0/0b6d5799bb726b24ad7fc7be720c170d8e497f575d02d47537de9a5bac074901/cache-1c80317fa3b1799d.arrow
[WARNING  | datasets.arrow_dataset]: Loading cached processed dataset at /root/.cache/huggingface/datasets/parquet/default-59b5a0749af325d9/0.0.0/0b6d5799bb726b24ad7fc7be720c170d8e497f575d02d47537de9a5bac074901/cache-bdd640fb06671ad1.arrow
[WARNING  | datasets.a

Train label counts: Counter({0: 5043, 1: 1371})
Class weights: [0.4275023639202118, 1.5724977254867554]


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Acc,Balanced Acc,Macro F1,F1 Biased,Prec Biased,Rec Biased,Tn,Fp,Fn,Tp
1,0.571900,0.513665,0.776808,0.775470,0.707217,0.564477,0.444444,0.773333,507,145,34,116
2,0.445400,0.510648,0.798005,0.788507,0.727474,0.588832,0.475410,0.773333,524,128,34,116
3,0.356200,0.626827,0.826683,0.772781,0.743348,0.597101,0.528205,0.686667,560,92,47,103


✅ Saved model to: bias_gate_deberta

🔧 Threshold tuning...


Best threshold: 0.55 F1_biased: 0.6053
Saved threshold.


In [23]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MISTRAL_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_m = AutoTokenizer.from_pretrained(MISTRAL_MODEL)
tokenizer_m.pad_token = tokenizer_m.eos_token

model_m = AutoModelForCausalLM.from_pretrained(
    MISTRAL_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_m.eval()

Loading checkpoint shards: 100%|██████████| 3/3 [00:11<00:00,  3.88s/it]


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
     

In [24]:
def mistral_rewrite(text, max_new_tokens=200):
    prompt = f"""
You are a professional neutral news editor.

Rewrite the following text in a neutral, objective journalistic tone.

Rules:
- Remove emotional or loaded language
- Preserve all factual information
- Do NOT add new facts
- Keep meaning intact

Text:
{text}

Neutral rewrite:
"""

    inputs = tokenizer_m(prompt, return_tensors="pt").to(model_m.device)

    with torch.no_grad():
        output = model_m.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer_m.eos_token_id
        )

    decoded = tokenizer_m.decode(output[0], skip_special_tokens=True)

    if "Neutral rewrite:" in decoded:
        return decoded.split("Neutral rewrite:")[-1].strip()
    else:
        return decoded.strip()

In [25]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

FACT_FILE = "outputs/factgate_fever_dev_2000.jsonl"
OUT_FILE  = "outputs/pass_with_bias_scores.jsonl"
BIAS_DIR  = "bias_gate_deberta"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Load Bias Gate
tokenizer = AutoTokenizer.from_pretrained(BIAS_DIR)
model = AutoModelForSequenceClassification.from_pretrained(BIAS_DIR).to(device)
model.eval()

# ---- Load threshold (optional)
try:
    with open(f"{BIAS_DIR}/gate_threshold.json") as f:
        threshold = json.load(f)["threshold"]
except:
    threshold = 0.5

print("Using threshold:", threshold)

# ---- Read only PASS rows
pass_rows = []
with open(FACT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r.get("decision") == "PASS":
            pass_rows.append(r)

print("PASS rows:", len(pass_rows))

# ---- Batch scoring
@torch.no_grad()
def score_batch(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[:, 1]  # P(biased)
    return probs.cpu().tolist()

BATCH_SIZE = 64 if torch.cuda.is_available() else 16
biased_count = 0

with open(OUT_FILE, "w", encoding="utf-8") as wf:
    for i in tqdm(range(0, len(pass_rows), BATCH_SIZE)):
        batch = pass_rows[i:i+BATCH_SIZE]
        texts = [(r.get("claim", "") or "")[:1500] for r in batch]

        scores = score_batch(texts)

        for r, p in zip(batch, scores):
            biased = p >= threshold
            if biased:
                biased_count += 1

            r2 = dict(r)
            r2["bias_score"] = {
                "prob_biased": float(p),
                "threshold": float(threshold),
                "biased": bool(biased)
            }

            wf.write(json.dumps(r2, ensure_ascii=False) + "\n")

print("Saved:", OUT_FILE)
print("Biased detected:", biased_count, "/", len(pass_rows))

Using threshold: 0.55
PASS rows: 5727


100%|██████████| 90/90 [00:03<00:00, 24.84it/s]

Saved: outputs/pass_with_bias_scores.jsonl
Biased detected: 143 / 5727


In [26]:
import json, os, re
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

# =========================
# Paths
# =========================
IN_FILE  = "outputs/pass_with_bias_scores.jsonl"
OUT_FILE = "outputs/ainsf_pass_final.jsonl"

# =========================
# Strict Guards (paper-level)
# =========================
SBERT_THRESHOLD   = 0.90
NLI_ENT_THRESHOLD = 0.70
MAX_RETRY         = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# Load PASS rows
# =========================
rows = []
with open(IN_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

biased_rows = [r for r in rows if r.get("bias_score", {}).get("biased", False)]
print("Total PASS rows:", len(rows))
print("Biased rows:", len(biased_rows))

# =========================
# Load Mistral
# =========================
GEN_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

tok = AutoTokenizer.from_pretrained(GEN_MODEL, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

gen = pipeline("text-generation", model=gen_model, tokenizer=tok)
print("✅ Loaded Mistral")

# =========================
# Load SBERT + NLI
# =========================
print("Loading SBERT...")
sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

print("Loading NLI...")
NLI_MODEL = "ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli"
nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)
nli_model.eval()
id2label = nli_model.config.id2label

def get_entailment_prob(premise, hypothesis):
    with torch.no_grad():
        enc = nli_tok(premise, hypothesis, truncation=True, max_length=512, return_tensors="pt").to(device)
        logits = nli_model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        labels = [id2label[i].lower() for i in range(len(probs))]
        for i, l in enumerate(labels):
            if "entail" in l:
                return float(probs[i])
    return 0.0

# =========================
# Additional HARD Guards
# =========================

def quantifier_mismatch(orig, rew):
    quantifiers = ["most", "some", "many", "all", "none"]
    for q in quantifiers:
        if q in orig.lower() and q not in rew.lower():
            return True
    return False

def entity_shift(orig, rew):
    orig_caps = set(re.findall(r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*", orig))
    rew_caps = set(re.findall(r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*", rew))
    return not orig_caps.issubset(rew_caps)

# =========================
# Prompt
# =========================
def build_prompt(text, strict=False):
    extra = ""
    if strict:
        extra = (
            "\nExtra STRICT rules:\n"
            "- Do NOT change quantifiers (most, some, many, all).\n"
            "- Do NOT weaken responsibility or intent.\n"
            "- Do NOT add roles/titles not present.\n"
            "- Do NOT change named entities.\n"
        )

    return (
        "<s>[INST] "
        "Rewrite the following text in a neutral, objective journalistic tone.\n"
        "Preserve all facts exactly. Remove only emotional wording.\n"
        f"{extra}\n"
        f"Text:\n{text} [/INST]"
    )

def clean_output(s):
    return re.sub(r"\s+", " ", (s or "")).strip()

# =========================
# Guarded Rewrite
# =========================
@torch.no_grad()
def rewrite_with_guard(original_text):
    last_sim, last_ent = 0.0, 0.0

    for attempt in range(MAX_RETRY + 1):

        prompt = build_prompt(original_text, strict=(attempt > 0))

        out = gen(
            prompt,
            max_new_tokens=120,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tok.eos_token_id,
            return_full_text=False
        )[0]["generated_text"]

        rewritten = clean_output(out)

        if len(rewritten) < 5:
            continue

        # SBERT
        emb = sbert.encode([original_text, rewritten], normalize_embeddings=True)
        sim = float(np.dot(emb[0], emb[1]))

        # NLI
        ent = float(get_entailment_prob(rewritten, original_text))

        last_sim, last_ent = sim, ent

        # HARD GUARDS
        if quantifier_mismatch(original_text, rewritten):
            continue
        if entity_shift(original_text, rewritten):
            continue

        if sim >= SBERT_THRESHOLD and ent >= NLI_ENT_THRESHOLD:
            return rewritten, sim, ent, "OK" if attempt == 0 else "RETRIED_OK"

    return original_text, last_sim, last_ent, "FALLBACK"

# =========================================================
# Rewrite ONLY biased rows
# =========================================================
rewrite_map = {}
rejected_examples = []

for r in tqdm(biased_rows, desc="Rewriting biased rows"):
    claim = (r.get("claim", "") or "").strip()
    if not claim:
        continue

    rewritten, sim, ent, status = rewrite_with_guard(claim)
    key = r.get("id", claim)
    rewrite_map[key] = (rewritten, sim, ent, status)

    if status == "FALLBACK":
        rejected_examples.append({
            "claim": claim,
            "last_similarity": round(sim,4),
            "last_entailment": round(ent,4)
        })

# =========================================================
# Merge Back
# =========================================================
results = []
counts = {"NOT_BIASED":0,"OK":0,"RETRIED_OK":0,"FALLBACK":0}

for r in rows:
    claim = (r.get("claim", "") or "").strip()
    key = r.get("id", claim)

    if key in rewrite_map:
        rewritten, sim, ent, status = rewrite_map[key]
    else:
        rewritten, sim, ent, status = claim, 1.0, 1.0, "NOT_BIASED"

    counts[status] += 1

    r2 = dict(r)
    r2["final_text"] = rewritten
    r2["rewrite_status"] = status
    r2["semantic_check"] = {
        "sbert_similarity": round(float(sim),4),
        "entailment_prob": round(float(ent),4)
    }
    results.append(r2)

# =========================
# Save
# =========================
os.makedirs("outputs", exist_ok=True)
with open(OUT_FILE, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("\n✅ Saved:", OUT_FILE)
print("Rewrite status breakdown:", counts)
print("Rejected (fallback) examples:", len(rejected_examples))

Device: cuda
Total PASS rows: 5727
Biased rows: 143


Loading checkpoint shards: 100%|██████████| 3/3 [00:11<00:00,  3.99s/it]


✅ Loaded Mistral
Loading SBERT...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loading NLI...


Some weights of the model checkpoint at ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temp


✅ Saved: outputs/ainsf_pass_final.jsonl
Rewrite status breakdown: {'NOT_BIASED': 5584, 'OK': 45, 'RETRIED_OK': 24, 'FALLBACK': 74}
Rejected (fallback) examples: 74


In [27]:
# -------------------------
# 2) BIAS GATE METRICS (Before/After)  ✅ FIXED
# -------------------------
def bias_gate_metrics(pass_bias_rows: List[Dict[str, Any]], final_rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    out = {"available": False}
    if not pass_bias_rows:
        out["note"] = "pass_with_bias_scores.jsonl not found or empty"
        return out

    # =========================
    # BEFORE (original bias)
    # =========================
    before_probs = []
    before_biased = 0
    threshold = None

    for r in pass_bias_rows:
        prob = safe_get(r, ["bias_score","prob_biased"], None)
        if prob is not None:
            before_probs.append(float(prob))

        b = safe_get(r, ["bias_score","biased"], None)
        if b is True:
            before_biased += 1

        if threshold is None:
            thr = safe_get(r, ["bias_score","threshold"], None)
            if thr is not None:
                threshold = float(thr)

    n_pass = len(pass_bias_rows)

    out["available"] = True
    out["pass_samples"] = n_pass
    out["before"] = {
        "mean_prob_biased": mean(before_probs),
        "median_prob_biased": median(before_probs),
        "min_prob_biased": minv(before_probs),
        "max_prob_biased": maxv(before_probs),
        "biased_count": before_biased,
        "biased_rate": before_biased/n_pass if n_pass else 0.0,
        "threshold_if_present": threshold
    }

    # =========================
    # AFTER (rewrite bias)
    # =========================
    if not final_rows:
        out["after_note"] = "outputs/ainsf_pass_final_with_after_bias.jsonl"
        return out

    rewritten_before = []
    rewritten_after  = []
    rewritten_statuses = {"OK":0,"RETRIED_OK":0,"FALLBACK":0,"NOT_BIASED":0}

    for r in final_rows:
        status = (r.get("rewrite_status","") or "").upper()
        if status in rewritten_statuses:
            rewritten_statuses[status] += 1

        if status in ("OK","RETRIED_OK","FALLBACK"):
            b0 = safe_get(r, ["bias_score","prob_biased"], None)
            b1 = safe_get(r, ["bias_after","prob_biased"], None)

            if b0 is not None:
                rewritten_before.append(float(b0))
            if b1 is not None:
                rewritten_after.append(float(b1))

    out["rewrite_status_breakdown"] = rewritten_statuses

    # =========================
    # Compute reduction ONLY if both available
    # =========================
    if rewritten_before and rewritten_after and len(rewritten_before) == len(rewritten_after):

        reductions = [b0 - b1 for b0, b1 in zip(rewritten_before, rewritten_after)]

        out["rewrite_effect"] = {
            "rewritten_pairs_with_after_prob": len(reductions),
            "mean_before": mean(rewritten_before),
            "mean_after": mean(rewritten_after),
            "mean_reduction": mean(reductions),
            "median_reduction": median(reductions),
            "min_reduction": minv(reductions),
            "max_reduction": maxv(reductions),
        }

    else:
        out["rewrite_effect_note"] = "bias_after not found — run add_bias_after script first"

    return out

In [28]:
import json, os
import numpy as np
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

FINAL_FILE = "outputs/ainsf_pass_final.jsonl"
SAVE_WORST = "outputs/preservation_worst5.jsonl"

SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
NLI_MODEL   = "ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -------------------------
# Load rows
# -------------------------
rows = []
with open(FINAL_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

# -------------------------
# Select ONLY accepted rewrites
# (your pipeline uses rewrite_status)
# -------------------------
pairs = []
for r in rows:
    status = r.get("rewrite_status", "")
    if status in ("OK", "RETRIED_OK"):   # accepted rewrites only
        orig = (r.get("claim", "") or "").strip()
        rew  = (r.get("final_text", "") or "").strip()
        if orig and rew:
            pairs.append((orig, rew, r))

print("Accepted rewritten pairs (OK + RETRIED_OK):", len(pairs))
assert len(pairs) > 0, "No accepted rewrites found. Check rewrite_status fields."

orig_texts = [p[0] for p in pairs]
rew_texts  = [p[1] for p in pairs]

# =========================================================
# 1) SBERT Similarity
# =========================================================
print("\n✅ Loading SBERT...")
sbert = SentenceTransformer(SBERT_MODEL, device=device)

# NOTE: SentenceTransformer .half() can cause issues sometimes
# keep it fp32 unless you're sure it's stable
orig_emb = sbert.encode(orig_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
rew_emb  = sbert.encode(rew_texts,  convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)

sims = (orig_emb * rew_emb).sum(axis=1)  # cosine since normalized

# =========================================================
# 2) NLI Entailment: rewrite ⇒ original
# premise = rewritten, hypothesis = original
# =========================================================
print("\n✅ Loading NLI model...")
tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)
nli.eval()
id2label = nli.config.id2label

def _idx_of(key: str):
    key = key.lower()
    for i in range(len(id2label)):
        if key in str(id2label[i]).lower():
            return i
    return None

IDX_ENT = _idx_of("entail")
IDX_CON = _idx_of("contrad")
IDX_NEU = _idx_of("neutral")
if IDX_ENT is None or IDX_CON is None or IDX_NEU is None:
    IDX_CON, IDX_NEU, IDX_ENT = 0, 1, 2

@torch.no_grad()
def nli_entailment_probs(premises, hypotheses, bs=24):
    out_ent = []
    out_top = []
    for i in tqdm(range(0, len(premises), bs), desc="NLI batches"):
        prem = premises[i:i+bs]
        hyp  = hypotheses[i:i+bs]
        enc = tok(prem, hyp, truncation=True, max_length=512, padding=True, return_tensors="pt").to(device)
        logits = nli(**enc).logits
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()

        out_ent.extend(probs[:, IDX_ENT].tolist())
        out_top.extend(probs.argmax(axis=1).tolist())

    return np.array(out_ent), np.array(out_top)

ent_probs, top_idx = nli_entailment_probs(rew_texts, orig_texts, bs=24 if device=="cuda" else 8)

top_labels = [id2label[int(i)] for i in top_idx]
ent_is_top = np.array([int(i)==IDX_ENT for i in top_idx], dtype=np.int32)

# =========================================================
# Report
# =========================================================
print("\n================= PRESERVATION REPORT (ACCEPTED ONLY) =================")
print(f"Pairs evaluated: {len(pairs)}\n")

print(f"SBERT cosine similarity: mean={sims.mean():.4f}  median={np.median(sims):.4f}  min={sims.min():.4f}")
print(f"SBERT >= 0.90: {(sims >= 0.90).mean():.3f}   |  SBERT >= 0.95: {(sims >= 0.95).mean():.3f}")

print()
print(f"NLI entailment prob (rewrite⇒orig): mean={ent_probs.mean():.4f}  median={np.median(ent_probs):.4f}  min={ent_probs.min():.4f}")
print(f"NLI top-label is ENTAILMENT: {ent_is_top.mean():.3f}")

# =========================================================
# Worst 5 and save for paper/debug
# =========================================================
worst = np.argsort(sims)[:5]
print("\n================= WORST 5 (by SBERT similarity) =================")

worst_rows = []
for j, idx in enumerate(worst, 1):
    orig, rew, r = pairs[idx]
    print("\n---", j, "---")
    print("SBERT:", round(float(sims[idx]), 4), "| EntProb:", round(float(ent_probs[idx]), 4), "| TopNLI:", top_labels[idx])
    print("ORIG:", orig)
    print("REWR:", rew)

    worst_rows.append({
        "rank": j,
        "id": r.get("id", None),
        "rewrite_status": r.get("rewrite_status", None),
        "bias_prob_before": r.get("bias_score", {}).get("prob_biased", None),
        "sbert": float(sims[idx]),
        "entail_prob": float(ent_probs[idx]),
        "top_nli": top_labels[idx],
        "orig": orig,
        "rew": rew,
    })

os.makedirs("outputs", exist_ok=True)
with open(SAVE_WORST, "w", encoding="utf-8") as f:
    for x in worst_rows:
        f.write(json.dumps(x, ensure_ascii=False) + "\n")

print("\n✅ Saved worst 5 pairs to:", SAVE_WORST)

Device: cuda
Accepted rewritten pairs (OK + RETRIED_OK): 69

✅ Loading SBERT...


Batches: 100%|██████████| 3/3 [00:00<00:00, 58.35it/s]



✅ Loading NLI model...


Some weights of the model checkpoint at ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
NLI batches: 100%|██████████| 3/3 [00:00<00:00, 33.29it/s]


================= PRESERVATION REPORT (ACCEPTED ONLY) =================
Pairs evaluated: 69

SBERT cosine similarity: mean=0.9389  median=0.9363  min=0.9002
SBERT >= 0.90: 1.000   |  SBERT >= 0.95: 0.362

NLI entailment prob (rewrite⇒orig): mean=0.9687  median=0.9859  min=0.7000
NLI top-label is ENTAILMENT: 1.000

================= WORST 5 (by SBERT similarity) =================

--- 1 ---
SBERT: 0.9002 | EntProb: 0.9691 | TopNLI: entailment
ORIG: Noel Fisher starred in Shameless and gained acclaim.
REWR: Noel Fisher appeared in the television show Shameless and received recognition for his role.

--- 2 ---
SBERT: 0.9006 | EntProb: 0.9918 | TopNLI: entailment
ORIG: Brad Wilk started his career as a clown.
REWR: Brad Wilk began his professional journey in the entertainment industry as a clown.

--- 3 ---
SBERT: 0.9031 | EntProb: 0.967 | TopNLI: entailment
ORIG: David Spade starred in Black Sheep and gained acclaim.
REWR: David Spade appeared in the film "Black Sheep" and received recog

In [39]:
# ============================
# AINSF_FINAL_METRICS.py ✅ (UPDATED)
# Auto-detects final file with after-bias probabilities
# Computes:
# - Fact Gate: PASS/DROP + label accuracy (if present)
# - Bias Gate: bias prevalence + BEFORE/AFTER bias reduction (if after probs exist)
# - Preservation: SBERT + NLI stats from semantic_check
# Saves: outputs/final_metrics.json + outputs/final_metrics.csv
# ============================

import os, json, math, statistics
from typing import Dict, Any, List

# -------------------------
# Paths (edit if needed)
# -------------------------
FACTGATE_FILE   = "outputs/factgate_fever_dev_2000.jsonl"
PASS_BIAS_FILE  = "outputs/pass_with_bias_scores.jsonl"

# AUTO pick final file
FINAL_FILE_CANDIDATES = [
    "outputs/ainsf_pass_final_with_after.jsonl",
]
FINAL_FILE = next((p for p in FINAL_FILE_CANDIDATES if os.path.exists(p)), FINAL_FILE_CANDIDATES[-1])

OUT_JSON = "outputs/final_metrics.json"
OUT_CSV  = "outputs/final_metrics.csv"

# -------------------------
# Helpers
# -------------------------
def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def _clean(xs: List[float]) -> List[float]:
    out = []
    for x in xs:
        if x is None:
            continue
        if isinstance(x, float) and math.isnan(x):
            continue
        out.append(float(x))
    return out

def mean(xs: List[float]) -> float:
    xs = _clean(xs)
    return float(sum(xs) / len(xs)) if xs else 0.0

def median(xs: List[float]) -> float:
    xs = _clean(xs)
    return float(statistics.median(xs)) if xs else 0.0

def minv(xs: List[float]) -> float:
    xs = _clean(xs)
    return float(min(xs)) if xs else 0.0

def maxv(xs: List[float]) -> float:
    xs = _clean(xs)
    return float(max(xs)) if xs else 0.0

def frac(mask: List[bool]) -> float:
    return float(sum(1 for x in mask if x) / len(mask)) if mask else 0.0

def safe_get(d: Dict[str, Any], keys: List[str], default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

# -------------------------
# 1) FACT GATE METRICS
# -------------------------
def fact_gate_metrics(fact_rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not fact_rows:
        return {"available": False, "note": "factgate file not found or empty"}

    def get_action(r):
        for path in [
            ["fact_gate","action"],
            ["gate","action"],
            ["decision"],
            ["action"],
            ["fact_gate_decision"],
        ]:
            v = safe_get(r, path, None)
            if isinstance(v, str):
                v = v.upper()
                if v in ("PASS","DROP"):
                    return v
        return None

    actions = [get_action(r) for r in fact_rows]
    actions = [a for a in actions if a in ("PASS","DROP")]

    total = len(actions) if actions else len(fact_rows)
    pass_n = sum(1 for a in actions if a == "PASS")
    drop_n = sum(1 for a in actions if a == "DROP")

    out = {
        "available": True,
        "samples": total,
        "pass": {"count": pass_n, "rate": pass_n/total if total else 0.0},
        "drop": {"count": drop_n, "rate": drop_n/total if total else 0.0},
    }

    def get_gold(r):
        for k in ["gold_label","label","fever_label"]:
            if isinstance(r.get(k), str):
                return r[k].upper()
        return None

    def get_pred(r):
        for k in ["pred_label","predicted_label","verifier_label","nli_label"]:
            if isinstance(r.get(k), str):
                return r[k].upper()
        return None

    golds, preds = [], []
    for r in fact_rows:
        g, p = get_gold(r), get_pred(r)
        if g and p:
            golds.append(g); preds.append(p)

    if golds:
        acc = sum(1 for g,p in zip(golds,preds) if g==p) / len(golds)
        out["verifier_label_accuracy_if_present"] = {"samples": len(golds), "acc": acc}

    return out

# -------------------------
# 2) BIAS GATE METRICS (Before/After)
# -------------------------
def bias_gate_metrics(pass_bias_rows: List[Dict[str, Any]], final_rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not pass_bias_rows:
        return {"available": False, "note": "pass_with_bias_scores.jsonl not found or empty"}

    # BEFORE
    before_probs = []
    before_biased = 0
    threshold = None

    for r in pass_bias_rows:
        prob = safe_get(r, ["bias_score","prob_biased"], None)
        if prob is not None:
            before_probs.append(float(prob))

        if safe_get(r, ["bias_score","biased"], None) is True:
            before_biased += 1

        if threshold is None:
            thr = safe_get(r, ["bias_score","threshold"], None)
            if thr is not None:
                threshold = float(thr)

    n_pass = len(pass_bias_rows)

    out = {
        "available": True,
        "pass_samples": n_pass,
        "before": {
            "mean_prob_biased": mean(before_probs),
            "median_prob_biased": median(before_probs),
            "min_prob_biased": minv(before_probs),
            "max_prob_biased": maxv(before_probs),
            "biased_count": before_biased,
            "biased_rate": before_biased/n_pass if n_pass else 0.0,
            "threshold_if_present": threshold,
        }
    }

    if not final_rows:
        out["after_note"] = "final file not found; cannot compute after-bias reduction"
        return out

    # AFTER: accept multiple possible locations
    AFTER_PATHS = [
        ["bias_after","prob_biased"],
        ["bias_after_prob"],
        ["bias_score_after","prob_biased"],
        ["bias_score_after_prob"],
        ["bias_after_score","prob_biased"],
        ["bias_after_score_prob"],
    ]

    after_probs_all = []
    rewritten_before = []
    rewritten_after  = []

    statuses = {"OK":0,"RETRIED_OK":0,"FALLBACK":0,"NOT_BIASED":0}

    def is_rewrite_row(r):
        st = (r.get("rewrite_status") or "").upper()
        if st in ("OK","RETRIED_OK","FALLBACK"):
            return True
        # fallback: some files may store rewrite.action
        act = safe_get(r, ["rewrite","action"], None)
        if isinstance(act, str) and act.upper() == "REWRITE":
            return True
        return False

    for r in final_rows:
        st = (r.get("rewrite_status") or "").upper()
        if st in statuses:
            statuses[st] += 1
        else:
            # if rewrite_status missing, infer
            statuses["NOT_BIASED"] += 1

        # overall after probs
        found_after = None
        for path in AFTER_PATHS:
            v = safe_get(r, path, None)
            if v is not None:
                found_after = float(v)
                after_probs_all.append(found_after)
                break

        # rewrite-only before/after
        if is_rewrite_row(r):
            b0 = safe_get(r, ["bias_score","prob_biased"], None)
            if b0 is not None:
                rewritten_before.append(float(b0))

            b1 = None
            for path in AFTER_PATHS:
                v = safe_get(r, path, None)
                if v is not None:
                    b1 = float(v); break
            if b1 is not None:
                rewritten_after.append(b1)

    out["rewrite_status_breakdown"] = statuses

    if after_probs_all:
        out["after"] = {
            "mean_prob_biased": mean(after_probs_all),
            "median_prob_biased": median(after_probs_all),
            "min_prob_biased": minv(after_probs_all),
            "max_prob_biased": maxv(after_probs_all),
        }
    else:
        out["after_note"] = f"No after-bias probability found in {FINAL_FILE}. Make sure your final file contains bias_after.prob_biased."

    if rewritten_before and rewritten_after and len(rewritten_before) == len(rewritten_after):
        reductions = [b0-b1 for b0,b1 in zip(rewritten_before, rewritten_after)]
        out["rewrite_effect"] = {
            "rewritten_pairs_with_after_prob": len(reductions),
            "mean_before": mean(rewritten_before),
            "mean_after": mean(rewritten_after),
            "mean_reduction": mean(reductions),
        }
    else:
        out["rewrite_effect_note"] = "Could not compute rewrite bias reduction (missing after bias probs for rewritten rows)."

    return out

# -------------------------
# 3) PRESERVATION METRICS
# -------------------------
def preservation_metrics(final_rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not final_rows:
        return {"available": False, "note": "final file not found or empty"}

    sims_all, ents_all = [], []
    sims_acc, ents_acc = [], []
    n_all, n_acc = 0, 0

    for r in final_rows:
        st = (r.get("rewrite_status","") or "").upper()
        sem = r.get("semantic_check", {}) if isinstance(r.get("semantic_check", {}), dict) else {}

        s = sem.get("sbert_similarity", None)
        e = sem.get("entailment_prob", None)

        if st in ("OK","RETRIED_OK","FALLBACK"):
            n_all += 1
            if s is not None: sims_all.append(float(s))
            if e is not None: ents_all.append(float(e))

        if st in ("OK","RETRIED_OK"):
            n_acc += 1
            if s is not None: sims_acc.append(float(s))
            if e is not None: ents_acc.append(float(e))

    def summarize(xs: List[float], thr1=0.90, thr2=0.95):
        return {
            "mean": mean(xs),
            "median": median(xs),
            "min": minv(xs),
            "max": maxv(xs),
            f"ge_{thr1}": frac([x >= thr1 for x in xs]),
            f"ge_{thr2}": frac([x >= thr2 for x in xs]),
        }

    out = {
        "available": True,
        "pairs_rewritten_any_status": n_all,
        "pairs_accepted_only": n_acc,
    }

    if sims_all: out["sbert_all_rewrites"] = summarize(sims_all)
    if ents_all: out["nli_entail_all_rewrites"] = summarize(ents_all, thr1=0.70, thr2=0.90)

    if sims_acc: out["sbert_accepted_only"] = summarize(sims_acc)
    if ents_acc: out["nli_entail_accepted_only"] = summarize(ents_acc, thr1=0.70, thr2=0.90)

    if sims_acc and ents_acc:
        out["ainsf_safety_score_accepted_only"] = (mean(sims_acc) + mean(ents_acc)) / 2.0

    return out

# -------------------------
# MAIN
# -------------------------
def main():
    os.makedirs("outputs", exist_ok=True)

    fact_rows      = read_jsonl(FACTGATE_FILE)
    pass_bias_rows = read_jsonl(PASS_BIAS_FILE)
    final_rows     = read_jsonl(FINAL_FILE)

    metrics = {
        "fact_gate": fact_gate_metrics(fact_rows),
        "bias_gate": bias_gate_metrics(pass_bias_rows, final_rows),
        "preservation": preservation_metrics(final_rows),
        "files": {
            "factgate_file": FACTGATE_FILE if os.path.exists(FACTGATE_FILE) else None,
            "pass_bias_file": PASS_BIAS_FILE if os.path.exists(PASS_BIAS_FILE) else None,
            "final_file": FINAL_FILE if os.path.exists(FINAL_FILE) else None,
        }
    }

    print("\n================ FINAL AINSF METRICS REPORT ================\n")
    print("📌 Using FINAL_FILE:", FINAL_FILE, "\n")

    # Fact gate
    fg = metrics["fact_gate"]
    if fg.get("available"):
        print("✅ FACT GATE")
        print(f"Samples: {fg['samples']}")
        print(f"PASS: {fg['pass']['count']} ({fg['pass']['rate']:.3f})")
        print(f"DROP: {fg['drop']['count']} ({fg['drop']['rate']:.3f})")
        if "verifier_label_accuracy_if_present" in fg:
            v = fg["verifier_label_accuracy_if_present"]
            print(f"Verifier label accuracy (if present): {v['acc']:.4f} on {v['samples']} samples")
        print()
    else:
        print("⚠️ FACT GATE:", fg.get("note",""), "\n")

    # Bias gate
    bg = metrics["bias_gate"]
    if bg.get("available"):
        b0 = bg["before"]
        print("✅ BIAS GATE (BEFORE)")
        print(f"PASS samples: {bg['pass_samples']}")
        print(f"Mean P(biased): {b0['mean_prob_biased']:.4f} | Median: {b0['median_prob_biased']:.4f}")
        print(f"Biased count: {b0['biased_count']} ({b0['biased_rate']:.3f})")
        if b0.get("threshold_if_present") is not None:
            print(f"Threshold: {b0['threshold_if_present']}")
        print()

        print("✅ REWRITE STATUS")
        print(bg.get("rewrite_status_breakdown", {}))
        print()

        if "rewrite_effect" in bg:
            reff = bg["rewrite_effect"]
            print("✅ BIAS REDUCTION (rewrite-only)")
            print(f"Pairs: {reff['rewritten_pairs_with_after_prob']}")
            print(f"Mean before: {reff['mean_before']:.4f}")
            print(f"Mean after : {reff['mean_after']:.4f}")
            print(f"Reduction  : {reff['mean_reduction']:.4f}")
            print()
        else:
            print("⚠️", bg.get("rewrite_effect_note",""), "\n")
    else:
        print("⚠️ BIAS GATE:", bg.get("note",""), "\n")

    # Preservation
    pr = metrics["preservation"]
    if pr.get("available"):
        print("✅ PRESERVATION (Semantic + Factual)")
        print(f"Pairs rewritten (any status): {pr['pairs_rewritten_any_status']}")
        print(f"Accepted only (OK/RETRIED_OK): {pr['pairs_accepted_only']}\n")

        if "sbert_accepted_only" in pr:
            s = pr["sbert_accepted_only"]
            print("SBERT (accepted):",
                  f"mean={s['mean']:.4f} median={s['median']:.4f} min={s['min']:.4f}",
                  f"| >=0.90={s['ge_0.9']:.3f} >=0.95={s['ge_0.95']:.3f}")
        if "nli_entail_accepted_only" in pr:
            e = pr["nli_entail_accepted_only"]
            print("NLI entail (accepted):",
                  f"mean={e['mean']:.4f} median={e['median']:.4f} min={e['min']:.4f}",
                  f"| >=0.70={e['ge_0.7']:.3f} >=0.90={e['ge_0.9']:.3f}")

        if "ainsf_safety_score_accepted_only" in pr:
            print("\nAINSF Safety Score (accepted):", f"{pr['ainsf_safety_score_accepted_only']:.4f}")
        print()
    else:
        print("⚠️ PRESERVATION:", pr.get("note",""), "\n")

    # Save JSON
    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print("✅ Saved:", OUT_JSON)

    # Save CSV summary
    try:
        import csv
        flat = {
            "final_file": FINAL_FILE,
            "fact_samples": fg.get("samples", 0) if fg.get("available") else 0,
            "fact_pass_rate": fg.get("pass", {}).get("rate", 0.0) if fg.get("available") else 0.0,
            "fact_drop_rate": fg.get("drop", {}).get("rate", 0.0) if fg.get("available") else 0.0,

            "bias_pass_samples": bg.get("pass_samples", 0) if bg.get("available") else 0,
            "bias_mean_before": bg.get("before", {}).get("mean_prob_biased", 0.0) if bg.get("available") else 0.0,
            "bias_biased_rate_before": bg.get("before", {}).get("biased_rate", 0.0) if bg.get("available") else 0.0,

            "rewrite_pairs_any": pr.get("pairs_rewritten_any_status", 0) if pr.get("available") else 0,
            "rewrite_pairs_accepted": pr.get("pairs_accepted_only", 0) if pr.get("available") else 0,
            "sbert_mean_accepted": pr.get("sbert_accepted_only", {}).get("mean", 0.0) if pr.get("available") else 0.0,
            "nli_ent_mean_accepted": pr.get("nli_entail_accepted_only", {}).get("mean", 0.0) if pr.get("available") else 0.0,
            "ainsf_safety_score": pr.get("ainsf_safety_score_accepted_only", 0.0) if pr.get("available") else 0.0,
        }

        # include bias reduction if available
        if bg.get("available") and "rewrite_effect" in bg:
            flat["bias_mean_before_rewrite"] = bg["rewrite_effect"]["mean_before"]
            flat["bias_mean_after_rewrite"]  = bg["rewrite_effect"]["mean_after"]
            flat["bias_mean_reduction"]      = bg["rewrite_effect"]["mean_reduction"]
        else:
            flat["bias_mean_before_rewrite"] = ""
            flat["bias_mean_after_rewrite"]  = ""
            flat["bias_mean_reduction"]      = ""

        with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(flat.keys()))
            w.writeheader()
            w.writerow(flat)

        print("✅ Saved:", OUT_CSV)
    except Exception as e:
        print("⚠️ Could not write CSV:", type(e).__name__, str(e)[:120])

    print("\n================ DONE ================\n")


if __name__ == "__main__":
    main()


================ FINAL AINSF METRICS REPORT ================

📌 Using FINAL_FILE: outputs/ainsf_pass_final_with_after.jsonl 

✅ FACT GATE
Samples: 10000
PASS: 5727 (0.573)
DROP: 4273 (0.427)
Verifier label accuracy (if present): 0.5889 on 10000 samples

✅ BIAS GATE (BEFORE)
PASS samples: 5727
Mean P(biased): 0.0659 | Median: 0.0237
Biased count: 143 (0.025)
Threshold: 0.55

✅ REWRITE STATUS
{'OK': 45, 'RETRIED_OK': 24, 'FALLBACK': 74, 'NOT_BIASED': 5584}

✅ BIAS REDUCTION (rewrite-only)
Pairs: 143
Mean before: 0.7580
Mean after : 0.6084
Reduction  : 0.1497

✅ PRESERVATION (Semantic + Factual)
Pairs rewritten (any status): 143
Accepted only (OK/RETRIED_OK): 69

SBERT (accepted): mean=0.9389 median=0.9363 min=0.9002 | >=0.90=1.000 >=0.95=0.362
NLI entail (accepted): mean=0.9687 median=0.9859 min=0.7002 | >=0.70=1.000 >=0.90=0.913

AINSF Safety Score (accepted): 0.9538

✅ Saved: outputs/final_metrics.json
✅ Saved: outputs/final_metrics.csv

================ DONE ================



In [41]:
# ============================
# AINSF_STATISTICAL_INFERENCE.py
# Proper statistical validation
# ============================

import json
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
import math

FINAL_FILE = "outputs/ainsf_pass_final_with_after.jsonl"

rows = []
with open(FINAL_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

# ------------------------------------------------
# Collect rewrite-only before/after bias probs
# ------------------------------------------------
before = []
after = []

for r in rows:
    status = (r.get("rewrite_status") or "").upper()
    if status in ("OK", "RETRIED_OK", "FALLBACK"):
        b0 = r.get("bias_score", {}).get("prob_biased", None)
        b1 = r.get("bias_after", {}).get("prob_biased", None)
        if b0 is not None and b1 is not None:
            before.append(float(b0))
            after.append(float(b1))

before = np.array(before)
after  = np.array(after)
diff   = before - after

print("Rewritten pairs:", len(before))

# ------------------------------------------------
# 1️⃣ Paired t-test
# ------------------------------------------------
t_stat, p_value = ttest_rel(before, after)

# ------------------------------------------------
# 2️⃣ Wilcoxon (non-parametric)
# ------------------------------------------------
w_stat, w_p = wilcoxon(before, after)

# ------------------------------------------------
# 3️⃣ Cohen's d (paired)
# ------------------------------------------------
cohens_d = diff.mean() / diff.std(ddof=1)

# ------------------------------------------------
# 4️⃣ 95% Confidence Interval for mean reduction
# ------------------------------------------------
mean_diff = diff.mean()
std_diff  = diff.std(ddof=1)
n = len(diff)

ci_low  = mean_diff - 1.96 * std_diff / math.sqrt(n)
ci_high = mean_diff + 1.96 * std_diff / math.sqrt(n)

# ------------------------------------------------
# 5️⃣ Safety score CI (accepted only)
# ------------------------------------------------
sbert = []
nli   = []

for r in rows:
    status = (r.get("rewrite_status") or "").upper()
    if status in ("OK","RETRIED_OK"):
        sem = r.get("semantic_check", {})
        if "sbert_similarity" in sem and "entailment_prob" in sem:
            sbert.append(sem["sbert_similarity"])
            nli.append(sem["entailment_prob"])

sbert = np.array(sbert)
nli   = np.array(nli)
safety = (sbert + nli) / 2

mean_safety = safety.mean()
std_safety  = safety.std(ddof=1)

ci_s_low  = mean_safety - 1.96 * std_safety / math.sqrt(len(safety))
ci_s_high = mean_safety + 1.96 * std_safety / math.sqrt(len(safety))

# ------------------------------------------------
# PRINT REPORT
# ------------------------------------------------
print("\n================ STATISTICAL INFERENCE ================\n")

print("Bias Reduction (Before vs After)")
print("-----------------------------------")
print(f"Mean Before: {before.mean():.4f}")
print(f"Mean After : {after.mean():.4f}")
print(f"Mean Reduction: {mean_diff:.4f}")
print(f"95% CI of Reduction: [{ci_low:.4f}, {ci_high:.4f}]")
print()
print("Paired t-test:")
print(f"t = {t_stat:.4f},  p = {p_value:.6f}")
print()
print("Wilcoxon signed-rank:")
print(f"W = {w_stat:.4f},  p = {w_p:.6f}")
print()
print("Effect Size (Cohen's d):", round(cohens_d,4))

if abs(cohens_d) >= 0.8:
    print("Effect: LARGE")
elif abs(cohens_d) >= 0.5:
    print("Effect: MEDIUM")
elif abs(cohens_d) >= 0.2:
    print("Effect: SMALL")
else:
    print("Effect: NEGLIGIBLE")

print("\nSafety Score (Accepted Only)")
print("-----------------------------------")
print(f"Mean Safety: {mean_safety:.4f}")
print(f"95% CI: [{ci_s_low:.4f}, {ci_s_high:.4f}]")

print("\n=======================================================")

Rewritten pairs: 143

================ STATISTICAL INFERENCE ================

Bias Reduction (Before vs After)
-----------------------------------
Mean Before: 0.7580
Mean After : 0.6084
Mean Reduction: 0.1497
95% CI of Reduction: [0.1073, 0.1921]

Paired t-test:
t = 6.9167,  p = 0.000000

Wilcoxon signed-rank:
W = 2560.0000,  p = 0.000000

Effect Size (Cohen's d): 0.5784
Effect: MEDIUM

Safety Score (Accepted Only)
-----------------------------------
Mean Safety: 0.9538
95% CI: [0.9476, 0.9601]



In [ ]:
# ===============================
# AINSF_ADVANCED_STATISTICS.py
# ===============================

import json
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.power import TTestPower
import random
import math

FINAL_FILE = "outputs/ainsf_pass_final_with_after.jsonl"

# Optional baseline file (if you have it)
SINGLE_LLM_FILE = "outputs/singlellm_pass_with_after_bias.jsonl"

# ----------------------------------------------------
# Load AINSF data
# ----------------------------------------------------
rows = []
with open(FINAL_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

before = []
after = []

before_bin = []
after_bin = []

for r in rows:
    status = (r.get("rewrite_status") or "").upper()
    if status in ("OK","RETRIED_OK","FALLBACK"):
        b0 = r.get("bias_score", {}).get("prob_biased")
        b1 = r.get("bias_after", {}).get("prob_biased")
        if b0 is not None and b1 is not None:
            before.append(float(b0))
            after.append(float(b1))

            before_bin.append(int(b0 >= 0.55))
            after_bin.append(int(b1 >= 0.55))

before = np.array(before)
after  = np.array(after)
diff   = before - after

n = len(diff)

print("Samples:", n)

# =====================================================
# 1️⃣ POST-HOC POWER ANALYSIS
# =====================================================
effect_size = diff.mean() / diff.std(ddof=1)
analysis = TTestPower()
power = analysis.power(effect_size=abs(effect_size), nobs=n, alpha=0.05)

# =====================================================
# 2️⃣ BOOTSTRAP CONFIDENCE INTERVAL (Bias Reduction)
# =====================================================
boot_means = []
B = 10000

for _ in range(B):
    sample = np.random.choice(diff, size=n, replace=True)
    boot_means.append(sample.mean())

boot_means = np.array(boot_means)
ci_boot_low  = np.percentile(boot_means, 2.5)
ci_boot_high = np.percentile(boot_means, 97.5)

# =====================================================
# 3️⃣ McNemar Test (classification change)
# =====================================================
# Contingency table:
#            After NonBiased   After Biased
# Before NB         a              b
# Before B          c              d

a=b=c=d=0
for b0,b1 in zip(before_bin, after_bin):
    if b0==0 and b1==0: a+=1
    if b0==0 and b1==1: b+=1
    if b0==1 and b1==0: c+=1
    if b0==1 and b1==1: d+=1

table = [[a,b],[c,d]]
mcnemar_result = mcnemar(table, exact=True)

# =====================================================
# 4️⃣ Compare against SingleLLM (if exists)
# =====================================================
single_mean = None
p_compare = None

try:
    single_rows = []
    with open(SINGLE_LLM_FILE, "r", encoding="utf-8") as f:
        for line in f:
            single_rows.append(json.loads(line))

    single_after = []
    for r in single_rows:
        b = r.get("bias_after", {}).get("prob_biased")
        if b is not None:
            single_after.append(float(b))

    single_after = np.array(single_after)
    single_mean = single_after.mean()

    from scipy.stats import ttest_ind
    t2, p2 = ttest_ind(after, single_after, equal_var=False)
    p_compare = p2

except:
    pass

# =====================================================
# PRINT RESULTS
# =====================================================
print("\n================ ADVANCED STATISTICS ================\n")

print("Post-hoc Power Analysis")
print("-------------------------")
print("Effect size (Cohen's d):", round(effect_size,4))
print("Power:", round(power,4))

print("\nBootstrap 95% CI (Bias Reduction)")
print("-----------------------------------")
print(f"[{ci_boot_low:.4f}, {ci_boot_high:.4f}]")

print("\nMcNemar Test (Bias Classification Change)")
print("------------------------------------------")
print("Contingency table:", table)
print("p-value:", mcnemar_result.pvalue)



In [46]:
# ============================
# BASELINE-1 (FEVER): Retrieval (Dense+BM25) + (optional) Rerank + NLI Verifier
# Uses your existing artifacts in fever_dense/
# NO bias gate, NO rewrite
# ============================

import os, json, math, sqlite3, re
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np
import torch
import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import CrossEncoder

# =========================
# PATHS (yours)
# =========================
DB_PATH     = "fever_data/fever_wiki.sqlite"
FEVER_DEV   = "fever_data/shared_task_dev.jsonl"
FEVER_TRAIN = "fever_data/train.jsonl"  # optional

OUT_DIR = "fever_dense"
os.makedirs(OUT_DIR, exist_ok=True)

INDEX_PATH = os.path.join(OUT_DIR, "faiss.index")
META_PATH  = os.path.join(OUT_DIR, "passage_meta.jsonl")
BM25_DB    = os.path.join(OUT_DIR, "bm25_passages.sqlite")

# =========================
# MODELS
# =========================
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
VERIFIER_MODEL = "ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE=="cuda" else None)

# =========================
# EVAL SETTINGS
# =========================
TOPK_FINAL = 10
MAX_EVIDENCE_FOR_VERIFIER = 5
EVAL_SAMPLES = 10000  # set 19998 for full dev

TOPK_DENSE_CAND = 50
TOPK_BM25_CAND  = 50

# Toggle hybrid + rerank
USE_BM25 = True
USE_RERANK = True

# Output
OUT_JSONL = "outputs/baseline1_fever_dev_pred.jsonl"
os.makedirs("outputs", exist_ok=True)

# =========================
# Utils
# =========================
def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                yield json.loads(line)

def load_meta(meta_path: str) -> List[Dict[str, Any]]:
    meta = []
    with open(meta_path, "r", encoding="utf-8") as f:
        for line in f:
            meta.append(json.loads(line))
    return meta

def normalize_claim(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    return s

def fever_label_map(label: str) -> str:
    # FEVER labels typically: SUPPORTS / REFUTES / NOT_ENOUGH_INFO
    if not label: 
        return "NOT_ENOUGH_INFO"
    label = label.upper()
    if "SUPPORT" in label: return "SUPPORTS"
    if "REFUTE" in label:  return "REFUTES"
    return "NOT_ENOUGH_INFO"

# =========================
# Load FAISS + Meta
# =========================
assert os.path.exists(INDEX_PATH), f"Missing FAISS index at {INDEX_PATH}"
assert os.path.exists(META_PATH),  f"Missing meta at {META_PATH}"

print("✅ Loading FAISS index...")
index = faiss.read_index(INDEX_PATH)

print("✅ Loading meta...")
meta = load_meta(META_PATH)
print("Meta passages:", len(meta), "| FAISS ntotal:", index.ntotal)
assert len(meta) == index.ntotal, "Meta and FAISS ntotal mismatch"

# =========================
# Dense Encoder (SBERT)
# =========================
print("✅ Loading SBERT:", EMB_MODEL)
sbert = SentenceTransformer(EMB_MODEL, device=DEVICE)

# =========================
# BM25 (optional)
# =========================
bm25_conn = None
if USE_BM25:
    assert os.path.exists(BM25_DB), f"Missing BM25 db at {BM25_DB}"
    print("✅ Opening BM25 DB...")
    bm25_conn = sqlite3.connect(BM25_DB)
    bm25_conn.row_factory = sqlite3.Row

def bm25_search(query: str, k: int = 20) -> List[int]:
    # expects table: passages(id INTEGER PRIMARY KEY, text TEXT, bm25 virtual table etc.)
    # Your builder likely created an FTS5 table named "passages_fts" or "passages".
    # We'll try common table names.
    if bm25_conn is None:
        return []
    q = query.replace('"', ' ')
    cur = bm25_conn.cursor()

    # Try FTS table names
    for tbl in ["passages_fts", "passages"]:
        try:
            rows = cur.execute(
                f"SELECT rowid as pid FROM {tbl} WHERE {tbl} MATCH ? LIMIT ?;",
                (q, k)
            ).fetchall()
            if rows:
                return [int(r["pid"]) for r in rows]
        except Exception:
            continue

    return []

# =========================
# Reranker (optional)
# =========================
reranker = None
if USE_RERANK:
    print("✅ Loading reranker:", RERANK_MODEL)
    reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)

# =========================
# Verifier (NLI)
# =========================
print("✅ Loading verifier:", VERIFIER_MODEL)
v_tok = AutoTokenizer.from_pretrained(VERIFIER_MODEL)
v_model = AutoModelForSequenceClassification.from_pretrained(VERIFIER_MODEL).to(DEVICE)
v_model.eval()
id2label = {int(k): v for k, v in v_model.config.id2label.items()} if isinstance(v_model.config.id2label, dict) else v_model.config.id2label

def label_indices():
    # find indices robustly
    ent_i = con_i = neu_i = None
    for i in range(len(id2label)):
        lab = str(id2label[i]).lower()
        if "entail" in lab: ent_i = i
        elif "contrad" in lab: con_i = i
        elif "neutral" in lab: neu_i = i
    # fallback common order
    if con_i is None or neu_i is None or ent_i is None:
        con_i, neu_i, ent_i = 0, 1, 2
    return con_i, neu_i, ent_i

IDX_CON, IDX_NEU, IDX_ENT = label_indices()

@torch.no_grad()
def verify_claim_with_evidence(claim: str, evid_texts: List[str]) -> Dict[str, Any]:
    """
    Aggregate evidence by max entail/contrad across evidence pieces.
    """
    if not evid_texts:
        return {"label": "NOT_ENOUGH_INFO", "probs": {"entail": 0.0, "contrad": 0.0, "neutral": 1.0}}

    # batch encode (premise=evidence, hypothesis=claim)
    enc = v_tok(
        evid_texts,
        [claim]*len(evid_texts),
        truncation=True,
        max_length=512,
        padding=True,
        return_tensors="pt"
    ).to(DEVICE)

    logits = v_model(**enc).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()

    # evidence-level max pooling
    ent_max = float(probs[:, IDX_ENT].max())
    con_max = float(probs[:, IDX_CON].max())
    neu_max = float(probs[:, IDX_NEU].max())

    # decision: pick best among entail/contrad/neutral
    if ent_max >= con_max and ent_max >= neu_max:
        pred = "SUPPORTS"
    elif con_max >= ent_max and con_max >= neu_max:
        pred = "REFUTES"
    else:
        pred = "NOT_ENOUGH_INFO"

    return {"label": pred, "probs": {"entail": ent_max, "contrad": con_max, "neutral": neu_max}}

# =========================
# Retrieval
# =========================
def dense_search(query: str, k: int = 20) -> List[int]:
    q_emb = sbert.encode([query], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype=np.float32)
    D, I = index.search(q_emb, k)
    return [int(x) for x in I[0].tolist() if int(x) >= 0]

def get_text(pid: int) -> str:
    # meta entries should include the actual passage text
    # Common keys: "text", "passage", "sentence"
    m = meta[pid]
    for key in ["text", "passage", "sentences", "sentence"]:
        if key in m and isinstance(m[key], str):
            return m[key]
    # fallback if you stored list of sentences
    if "sents" in m and isinstance(m["sents"], list):
        return " ".join(m["sents"])
    return ""

def retrieve_candidates(claim: str) -> List[int]:
    dense_ids = dense_search(claim, TOPK_DENSE_CAND)
    ids = list(dense_ids)

    if USE_BM25:
        bm25_ids = bm25_search(claim, TOPK_BM25_CAND)
        # merge unique, keep dense first then bm25 extras
        seen = set(ids)
        for x in bm25_ids:
            if x not in seen:
                ids.append(x); seen.add(x)

    return ids

def rerank(claim: str, cand_ids: List[int], topk: int) -> List[int]:
    if not USE_RERANK or reranker is None:
        return cand_ids[:topk]

    texts = [get_text(i) for i in cand_ids]
    pairs = [[claim, t] for t in texts]
    scores = reranker.predict(pairs, batch_size=32)
    order = np.argsort(-np.asarray(scores))  # desc
    out = [cand_ids[int(i)] for i in order[:topk]]
    return out

# =========================
# Run evaluation
# =========================
dev_rows = list(iter_jsonl(FEVER_DEV))
if EVAL_SAMPLES and EVAL_SAMPLES < len(dev_rows):
    dev_rows = dev_rows[:EVAL_SAMPLES]

print("✅ FEVER dev samples:", len(dev_rows))

correct = 0
preds = []
for r in tqdm(dev_rows, desc="Baseline-1 eval"):
    claim = normalize_claim(r.get("claim", ""))
    gold  = fever_label_map(r.get("label", ""))

    cand_ids = retrieve_candidates(claim)
    top_ids = rerank(claim, cand_ids, TOPK_FINAL)

    evid_texts = [get_text(i) for i in top_ids][:MAX_EVIDENCE_FOR_VERIFIER]
    ver = verify_claim_with_evidence(claim, evid_texts)
    pred = ver["label"]

    correct += int(pred == gold)

    preds.append({
        "id": r.get("id", None),
        "claim": claim,
        "gold_label": gold,
        "pred_label": pred,
        "verifier_probs": ver["probs"],
        "evidence_pids": top_ids[:TOPK_FINAL],
        "evidence_texts": evid_texts,
    })

acc = correct / len(dev_rows) if dev_rows else 0.0
print(f"\n✅ Baseline-1 Label Accuracy on FEVER dev: {acc:.4f} (N={len(dev_rows)})")

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for x in preds:
        f.write(json.dumps(x, ensure_ascii=False) + "\n")

print("✅ Saved:", OUT_JSONL)

DEVICE: cuda | GPU: NVIDIA B200 MIG 1g.45gb
✅ Loading FAISS index...
✅ Loading meta...
Meta passages: 6473073 | FAISS ntotal: 6473073
✅ Loading SBERT: sentence-transformers/all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Opening BM25 DB...
✅ Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
✅ Loading verifier: ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli


Some weights of the model checkpoint at ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✅ FEVER dev samples: 10000


Baseline-1 eval: 100%|██████████| 10000/10000 [10:59<00:00, 15.17it/s]



✅ Baseline-1 Label Accuracy on FEVER dev: 0.6447 (N=10000)
✅ Saved: outputs/baseline1_fever_dev_pred.jsonl


In [44]:
# ============================
# BASELINE-2 ✅
# FACT GATE + BIAS GATE (NO REWRITE)
# Output = publish only if:
#   fact_gate == PASS  AND  bias_gate == NOT_BIASED
# Otherwise drop.
# ============================

import os, json, math
from typing import Dict, Any, List, Optional

import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# -------------------------
# INPUTS
# -------------------------
# Use your factgate output (2000 or 5000)
FACTGATE_FILE = "outputs/factgate_fever_dev_2000.jsonl"  # change if your file name differs
# If you already computed bias scores for PASS rows, point here:
PASS_BIAS_FILE = "outputs/pass_with_bias_scores.jsonl"   # optional shortcut

# If PASS_BIAS_FILE missing OR you want recompute bias:
BIAS_MODEL = "YOUR_BASIL_DEBERTA_BIAS_MODEL"  # <-- put your trained model path or HF id
BIAS_THRESHOLD = 0.55

# Output
OUT_FILE = "outputs/baseline2_fact_bias_gate_only.jsonl"
os.makedirs("outputs", exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# -------------------------
# Helpers
# -------------------------
def read_jsonl(path: str) -> List[Dict[str, Any]]:
    if not os.path.exists(path):
        return []
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                out.append(json.loads(line))
    return out

def safe_get(d: Dict[str, Any], keys: List[str], default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def get_fact_action(r: Dict[str, Any]) -> Optional[str]:
    # robustly detect PASS/DROP field
    for path in [
        ["fact_gate","action"],
        ["gate","action"],
        ["decision"],
        ["action"],
        ["fact_gate_decision"],
    ]:
        v = safe_get(r, path, None)
        if isinstance(v, str):
            v = v.upper()
            if v in ("PASS","DROP"):
                return v
    return None

def get_claim_text(r: Dict[str, Any]) -> str:
    # prefer claim field; fallback final_text/text
    for k in ["claim", "text", "final_text"]:
        if isinstance(r.get(k), str) and r.get(k).strip():
            return r[k].strip()
    return ""

# -------------------------
# Load FactGate rows
# -------------------------
fact_rows = read_jsonl(FACTGATE_FILE)
assert fact_rows, f"❌ Missing or empty FACTGATE_FILE: {FACTGATE_FILE}"

fact_actions = [get_fact_action(r) for r in fact_rows]
fact_actions_clean = [a for a in fact_actions if a in ("PASS","DROP")]
N = len(fact_actions_clean) if fact_actions_clean else len(fact_rows)

fact_pass = [r for r in fact_rows if get_fact_action(r) == "PASS"]
fact_drop = [r for r in fact_rows if get_fact_action(r) == "DROP"]

print("\n✅ FACT GATE")
print("Samples:", N)
print("PASS:", len(fact_pass), f"({len(fact_pass)/N:.3f})")
print("DROP:", len(fact_drop), f"({len(fact_drop)/N:.3f})")

# -------------------------
# If you already have PASS rows with bias_score, use that (fast)
# else compute bias_score now
# -------------------------
pass_bias_rows = []
if os.path.exists(PASS_BIAS_FILE):
    pass_bias_rows = read_jsonl(PASS_BIAS_FILE)
    # quick sanity: ensure it looks like PASS set
    if pass_bias_rows and safe_get(pass_bias_rows[0], ["bias_score","prob_biased"], None) is not None:
        print("\n✅ Using existing PASS_BIAS_FILE:", PASS_BIAS_FILE)
    else:
        pass_bias_rows = []

# map by id/claim to merge bias back
def row_key(r: Dict[str, Any]) -> str:
    if r.get("id") is not None:
        return str(r["id"])
    c = get_claim_text(r)
    return c

# If PASS_BIAS_FILE is not available/valid -> compute bias
bias_tok = None
bias_model = None

@torch.no_grad()
def bias_prob(texts: List[str], bs: int = 32) -> List[float]:
    # returns P(biased) assuming label mapping has "biased" label
    probs_out = []
    for i in range(0, len(texts), bs):
        batch = texts[i:i+bs]
        enc = bias_tok(batch, truncation=True, max_length=512, padding=True, return_tensors="pt").to(DEVICE)
        logits = bias_model(**enc).logits
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()

        # find biased index robustly
        id2label = bias_model.config.id2label
        biased_idx = None
        for j in range(probs.shape[1]):
            lab = str(id2label[j]).lower()
            if "bias" in lab and "not" not in lab:
                biased_idx = j
                break
        if biased_idx is None:
            # fallback assume class-1 is biased (common in binary)
            biased_idx = 1 if probs.shape[1] > 1 else 0

        probs_out.extend(probs[:, biased_idx].tolist())
    return probs_out

# Build bias map either from file or by computing
bias_map = {}

if pass_bias_rows:
    for r in pass_bias_rows:
        k = row_key(r)
        p = safe_get(r, ["bias_score","prob_biased"], None)
        b = safe_get(r, ["bias_score","biased"], None)
        if p is None:
            continue
        bias_map[k] = {"prob_biased": float(p), "biased": bool(b) if b is not None else (float(p) >= BIAS_THRESHOLD)}
else:
    print("\n⚠️ PASS_BIAS_FILE not usable. Computing bias from model:", BIAS_MODEL)
    assert BIAS_MODEL != "YOUR_BASIL_DEBERTA_BIAS_MODEL", "❌ Set BIAS_MODEL to your trained DeBERTa bias model path/ID."
    bias_tok = AutoTokenizer.from_pretrained(BIAS_MODEL, use_fast=True)
    bias_model = AutoModelForSequenceClassification.from_pretrained(BIAS_MODEL).to(DEVICE)
    bias_model.eval()

    pass_texts = [get_claim_text(r) for r in fact_pass]
    probs = bias_prob(pass_texts, bs=48 if DEVICE=="cuda" else 16)

    for r, p in zip(fact_pass, probs):
        k = row_key(r)
        bias_map[k] = {"prob_biased": float(p), "biased": float(p) >= BIAS_THRESHOLD}

# -------------------------
# Baseline-2 decision: PASS if (fact PASS) and (bias NOT_BIASED)
# -------------------------
published = []
dropped_bias = []

for r in fact_pass:
    k = row_key(r)
    c = get_claim_text(r)
    binfo = bias_map.get(k, {"prob_biased": None, "biased": None})

    prob = binfo["prob_biased"]
    biased = binfo["biased"]
    if biased is True:
        dropped_bias.append(r)
        continue

    out = dict(r)
    out["baseline2"] = {
        "fact_gate": "PASS",
        "bias_gate": "PASS",
        "bias_threshold": BIAS_THRESHOLD,
        "prob_biased": prob,
        "action": "PUBLISH_NO_REWRITE"
    }
    out["final_text"] = c  # unchanged
    published.append(out)

# -------------------------
# Save
# -------------------------
with open(OUT_FILE, "w", encoding="utf-8") as f:
    for r in published:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# -------------------------
# Report
# -------------------------
print("\n✅ BIAS GATE (ON FACT-PASS)")
print("Fact-PASS:", len(fact_pass))
print("Bias-DROP:", len(dropped_bias), f"({len(dropped_bias)/len(fact_pass):.3f})" if fact_pass else "")
print("Publish:", len(published), f"({len(published)/len(fact_pass):.3f})" if fact_pass else "")

print("\n✅ BASELINE-2 OUTPUT")
print("Saved:", OUT_FILE)

DEVICE: cuda

✅ FACT GATE
Samples: 10000
PASS: 5727 (0.573)
DROP: 4273 (0.427)

✅ Using existing PASS_BIAS_FILE: outputs/pass_with_bias_scores.jsonl

✅ BIAS GATE (ON FACT-PASS)
Fact-PASS: 5727
Bias-DROP: 143 (0.025)
Publish: 5584 (0.975)

✅ BASELINE-2 OUTPUT
Saved: outputs/baseline2_fact_bias_gate_only.jsonl


In [48]:
from sklearn.metrics import classification_report
import json

y_true = []
y_pred = []

with open("outputs/factgate_fever_dev_2000.jsonl") as f:
    for line in f:
        r = json.loads(line)
        y_true.append(r["gold_gate"])
        y_pred.append(r["decision"])

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

        DROP       0.93      0.60      0.73      6667
        PASS       0.53      0.91      0.67      3333

    accuracy                           0.70     10000
   macro avg       0.73      0.75      0.70     10000
weighted avg       0.80      0.70      0.71     10000



In [49]:
import json
import numpy as np

scores = []

with open("outputs/pass_with_bias_scores.jsonl") as f:
    for line in f:
        r = json.loads(line)
        scores.append(r["bias_score"]["prob_biased"])

print("Mean bias probability:", np.mean(scores))

Mean bias probability: 0.0658529112692802


In [1]:
#!/usr/bin/env python3
"""
BASELINE_RAG_LLM_BIASGATE.py ✅

Baseline: Retriever evidence (already in rows) + LLM generation + Bias Gate.

Input:
  - A JSONL where each row has at least:
      id, claim, retrieved
  - Example: outputs/factgate_fever_dev_2000.jsonl (your file)

Pipeline:
  claim -> evidence_snippet from retrieved -> LLM generates short neutral report -> bias gate scores generated text

Outputs:
  outputs/baseline_rag_llm_biasgate.jsonl
  outputs/baseline_rag_llm_biasgate_metrics.json

Notes:
- This is a BASELINE, not your full AINSF.
- No fact verification, no rewriting safety check.
"""

import os, json, math, statistics
from typing import Any, Dict, List

import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM

# -------------------------
# Paths / Config
# -------------------------
IN_FILE   = "outputs/factgate_fever_dev_2000.jsonl"
OUT_JSONL = "outputs/baseline_rag_llm_biasgate.jsonl"
OUT_METR  = "outputs/baseline_rag_llm_biasgate_metrics.json"

# Bias Gate (DeBERTa)
BIAS_DIR  = "bias_gate_deberta"
BIAS_MAX_LEN = 256
BIAS_THRESHOLD = 0.5

# LLM (Mistral) - set to your local path or HF id that you already used successfully
LLM_ID_OR_PATH = "mistralai/Mistral-7B-Instruct-v0.2"  # change if your env uses a local dir
LLM_MAX_NEW_TOKENS = 80
LLM_TEMPERATURE = 0.0  # deterministic baseline
LLM_TOP_P = 1.0

# Evidence snippet control
MAX_EVIDENCE_CHARS = 900

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# IO helpers
# -------------------------
def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: str, rows: List[Dict[str, Any]]) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as w:
        for r in rows:
            w.write(json.dumps(r, ensure_ascii=False) + "\n")

# -------------------------
# Stats helpers
# -------------------------
def _clean(xs: List[float]) -> List[float]:
    out = []
    for x in xs:
        if x is None:
            continue
        try:
            xf = float(x)
        except Exception:
            continue
        if isinstance(xf, float) and math.isnan(xf):
            continue
        out.append(xf)
    return out

def mean(xs: List[float]) -> float:
    xs = _clean(xs)
    return float(sum(xs) / len(xs)) if xs else 0.0

def median(xs: List[float]) -> float:
    xs = _clean(xs)
    return float(statistics.median(xs)) if xs else 0.0

def minv(xs: List[float]) -> float:
    xs = _clean(xs)
    return float(min(xs)) if xs else 0.0

def maxv(xs: List[float]) -> float:
    xs = _clean(xs)
    return float(max(xs)) if xs else 0.0

# -------------------------
# Retrieval -> evidence snippet
# -------------------------
def build_evidence_snippet(retrieved: Any) -> str:
    if retrieved is None:
        return ""
    texts: List[str] = []

    if isinstance(retrieved, list):
        for item in retrieved:
            if isinstance(item, dict):
                for k in ("text", "sentence", "passage", "evidence"):
                    if isinstance(item.get(k), str) and item[k].strip():
                        texts.append(item[k].strip())
                        break
                if isinstance(item.get("lines"), list):
                    joined = " ".join([str(x).strip() for x in item["lines"] if str(x).strip()])
                    if joined:
                        texts.append(joined)
            elif isinstance(item, str) and item.strip():
                texts.append(item.strip())

    elif isinstance(retrieved, dict):
        for k in ("text", "sentence", "passage", "evidence"):
            if isinstance(retrieved.get(k), str) and retrieved[k].strip():
                texts.append(retrieved[k].strip())
                break
        if isinstance(retrieved.get("lines"), list):
            joined = " ".join([str(x).strip() for x in retrieved["lines"] if str(x).strip()])
            if joined:
                texts.append(joined)

    snippet = " ".join(texts).strip()
    if len(snippet) > MAX_EVIDENCE_CHARS:
        snippet = snippet[:MAX_EVIDENCE_CHARS].rstrip() + "..."
    return snippet

# -------------------------
# Bias scoring (DeBERTa)
# -------------------------
def prob_biased_from_logits(logits: torch.Tensor) -> float:
    if logits.dim() == 1:
        logits = logits.unsqueeze(0)
    C = logits.size(-1)
    if C == 1:
        return float(torch.sigmoid(logits[0, 0]).item())
    probs = F.softmax(logits, dim=-1)[0]
    return float(probs[1].item())  # assume index 1 = biased

@torch.inference_mode()
def score_bias_prob(text: str, tokenizer, model, device) -> float:
    enc = tokenizer(
        text,
        truncation=True,
        max_length=BIAS_MAX_LEN,
        padding=True,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc)
    return prob_biased_from_logits(out.logits)

# -------------------------
# LLM generation
# -------------------------
def make_rag_prompt(claim: str, evidence: str) -> str:
    # keep it simple: a "neutral report" from evidence only
    return (
        "You are a neutral news writer.\n"
        "Write ONE short, factual, neutral sentence based ONLY on the evidence.\n"
        "Do NOT add opinions, emotion, or political framing.\n"
        "If evidence is insufficient, say: \"Insufficient evidence.\"\n\n"
        f"CLAIM: {claim}\n"
        f"EVIDENCE: {evidence}\n\n"
        "OUTPUT:"
    )

@torch.inference_mode()
def llm_generate(prompt: str, tok, mdl, device) -> str:
    inputs = tok(prompt, return_tensors="pt").to(device)
    gen = mdl.generate(
        **inputs,
        max_new_tokens=LLM_MAX_NEW_TOKENS,
        do_sample=(LLM_TEMPERATURE > 0.0),
        temperature=LLM_TEMPERATURE if LLM_TEMPERATURE > 0.0 else None,
        top_p=LLM_TOP_P,
        pad_token_id=tok.eos_token_id,
        eos_token_id=tok.eos_token_id,
    )
    out = tok.decode(gen[0], skip_special_tokens=True)

    # return only the part after "OUTPUT:"
    if "OUTPUT:" in out:
        out = out.split("OUTPUT:", 1)[-1].strip()
    return out.strip()

# -------------------------
# Main
# -------------------------
def main():
    os.makedirs("outputs", exist_ok=True)

    rows = read_jsonl(IN_FILE)
    if not rows:
        raise FileNotFoundError(f"Input not found/empty: {IN_FILE}")

    print("Device:", DEVICE)

    print("Loading LLM:", LLM_ID_OR_PATH)
    llm_tok = AutoTokenizer.from_pretrained(LLM_ID_OR_PATH, use_fast=True)
    llm_mdl = AutoModelForCausalLM.from_pretrained(
        LLM_ID_OR_PATH,
        torch_dtype=torch.float16 if DEVICE.startswith("cuda") else None,
        device_map="auto" if DEVICE.startswith("cuda") else None
    )
    llm_mdl.eval()

    print("Loading Bias model:", BIAS_DIR)
    bias_tok = AutoTokenizer.from_pretrained(BIAS_DIR)
    bias_mdl = AutoModelForSequenceClassification.from_pretrained(BIAS_DIR).to(DEVICE)
    bias_mdl.eval()

    probs = []
    biased_count = 0
    out_rows = []

    for r in tqdm(rows, desc="Baseline RAG+LLM+BiasGate"):
        claim = r.get("claim", "") or ""
        evidence = build_evidence_snippet(r.get("retrieved", None))

        prompt = make_rag_prompt(claim, evidence if evidence else "NO_EVIDENCE")
        gen_text = llm_generate(prompt, llm_tok, llm_mdl, DEVICE)

        p = score_bias_prob(gen_text, bias_tok, bias_mdl, DEVICE)
        is_biased = p >= BIAS_THRESHOLD
        probs.append(p)
        if is_biased:
            biased_count += 1

        out_rows.append({
            "id": r.get("id"),
            "claim": claim,
            "evidence_snippet": evidence if evidence else None,
            "rag_output": gen_text,
            "bias_score": {
                "prob_biased": float(p),
                "biased": bool(is_biased),
                "threshold": float(BIAS_THRESHOLD),
            }
        })

    write_jsonl(OUT_JSONL, out_rows)
    print("✅ Saved:", OUT_JSONL)

    metrics = {
        "input_file": IN_FILE,
        "n_samples": len(out_rows),
        "bias_threshold": BIAS_THRESHOLD,
        "prob_biased": {
            "mean": mean(probs),
            "median": median(probs),
            "min": minv(probs),
            "max": maxv(probs),
        },
        "biased": {
            "count": biased_count,
            "rate": biased_count / len(out_rows) if out_rows else 0.0,
        }
    }

    with open(OUT_METR, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print("✅ Saved:", OUT_METR)

    print("\n=== BASELINE SUMMARY (RAG+LLM+BiasGate) ===")
    print("Samples:", metrics["n_samples"])
    print("Biased rate:", f"{metrics['biased']['rate']:.4f}", f"({metrics['biased']['count']})")
    print("Mean P(biased):", f"{metrics['prob_biased']['mean']:.4f}")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Loading LLM: mistralai/Mistral-7B-Instruct-v0.2


Loading checkpoint shards: 100%|██████████| 3/3 [00:10<00:00,  3.53s/it]


Loading Bias model: bias_gate_deberta


Baseline RAG+LLM+BiasGate: 100%|██████████| 10000/10000 [1:35:03<00:00,  1.75it/s]


✅ Saved: outputs/baseline_rag_llm_biasgate.jsonl
✅ Saved: outputs/baseline_rag_llm_biasgate_metrics.json

=== BASELINE SUMMARY (RAG+LLM+BiasGate) ===
Samples: 10000
Biased rate: 0.0377 (377)
Mean P(biased): 0.0842


In [2]:
import json
import numpy as np
from scipy import stats

BASELINE_FILE = "outputs/baseline_rag_llm_biasgate.jsonl"
AINSF_FILE    = "outputs/ainsf_pass_final_with_after.jsonl"

def read_jsonl(path):
    rows = []
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def extract_probs(rows, key_path):
    out = []
    for r in rows:
        cur = r
        for k in key_path:
            cur = cur.get(k, None)
            if cur is None:
                break
        if cur is not None:
            out.append(float(cur))
    return np.array(out)

# Load
base_rows = read_jsonl(BASELINE_FILE)
ainsf_rows = read_jsonl(AINSF_FILE)

# Extract probabilities
baseline_probs = extract_probs(base_rows, ["bias_score","prob_biased"])
ainsf_after_probs = extract_probs(ainsf_rows, ["bias_after","prob_biased"])

# Make sure same length
n = min(len(baseline_probs), len(ainsf_after_probs))
baseline_probs = baseline_probs[:n]
ainsf_after_probs = ainsf_after_probs[:n]

print("Samples compared:", n)

# -------------------------
# Paired t-test
# -------------------------
t_stat, t_p = stats.ttest_rel(baseline_probs, ainsf_after_probs)

print("\nPaired t-test:")
print("t =", t_stat)
print("p =", t_p)

# -------------------------
# Wilcoxon signed-rank
# -------------------------
w_stat, w_p = stats.wilcoxon(baseline_probs, ainsf_after_probs)

print("\nWilcoxon signed-rank:")
print("W =", w_stat)
print("p =", w_p)

# -------------------------
# Effect size (Cohen's d)
# -------------------------
diff = baseline_probs - ainsf_after_probs
cohens_d = diff.mean() / diff.std(ddof=1)

print("\nEffect size (Cohen's d):", cohens_d)

# -------------------------
# Bootstrap 95% CI
# -------------------------
def bootstrap_ci(data, n_boot=5000):
    means = []
    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        means.append(sample.mean())
    return np.percentile(means, [2.5, 97.5])

ci = bootstrap_ci(diff)
print("\n95% CI of mean difference:", ci)

Samples compared: 5727

Paired t-test:
t = 9.135663366470377
p = 8.849002011579302e-20

Wilcoxon signed-rank:
W = 6753884.0
p = 7.763511213565513e-31

Effect size (Cohen's d): 0.12071924111214981

95% CI of mean difference: [0.01835316 0.02833172]


In [4]:
#!/usr/bin/env python3
"""
ACL_VIZ_BASELINE_VS_AINSF.py ✅ (ACL-ready)

Main-paper plots (keeps only strongest):
A) ECDF overlay
B) Paired difference histogram (Baseline - AINSF)
C) Boxplot + sampled points
D) Mean with 95% bootstrap CI (bars)

Removes:
- Scatter plot (often confusing + not needed)

Outputs:
  outputs/figs_acl/
    figA_ecdf.pdf/png
    figB_diff_hist.pdf/png
    figC_box_strip.pdf/png
    figD_mean_ci.pdf/png
    figure_stats.json

Tip for ACL main paper:
- Use (A) + (B) as a 2-panel combined figure if space is tight.
"""

import os, json
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Files
# -------------------------
BASELINE_FILE = "outputs/baseline_rag_llm_biasgate.jsonl"
AINSF_FILE    = "outputs/ainsf_pass_final_with_after.jsonl"

OUT_DIR = "outputs/figs_acl"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# ACL-style matplotlib defaults
# -------------------------
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "pdf.fonttype": 42,   # better PDF text embedding
    "ps.fonttype": 42,
})

# -------------------------
# Utils
# -------------------------
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def extract_baseline_prob(rows):
    return np.array([float(r["bias_score"]["prob_biased"]) for r in rows], dtype=float)

def extract_ainsf_after_prob(rows):
    return np.array([float(r["bias_after"]["prob_biased"]) for r in rows], dtype=float)

def savefig(name):
    pdf = os.path.join(OUT_DIR, f"{name}.pdf")
    png = os.path.join(OUT_DIR, f"{name}.png")
    plt.savefig(pdf, bbox_inches="tight")
    plt.savefig(png, dpi=300, bbox_inches="tight")
    print("✅ Saved:", pdf)
    print("✅ Saved:", png)

def ecdf(x):
    x = np.sort(x)
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y

def bootstrap_ci_mean(x, n_boot=8000, seed=7):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    means = []
    for _ in range(n_boot):
        samp = rng.choice(x, size=len(x), replace=True)
        means.append(float(samp.mean()))
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(lo), float(hi)

def bootstrap_ci_mean_diff(x, y, n_boot=8000, seed=7):
    # paired bootstrap
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = min(len(x), len(y))
    x, y = x[:n], y[:n]
    diffs = x - y
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots.append(float(diffs[idx].mean()))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(lo), float(hi)

# -------------------------
# Load + align
# -------------------------
base_rows = read_jsonl(BASELINE_FILE)
ainsf_rows = read_jsonl(AINSF_FILE)

baseline = extract_baseline_prob(base_rows)
ainsf_after = extract_ainsf_after_prob(ainsf_rows)

n = min(len(baseline), len(ainsf_after))
baseline = baseline[:n]
ainsf_after = ainsf_after[:n]
diff = baseline - ainsf_after

base_mean = float(baseline.mean())
ainsf_mean = float(ainsf_after.mean())
diff_mean = float(diff.mean())

base_ci = bootstrap_ci_mean(baseline)
ainsf_ci = bootstrap_ci_mean(ainsf_after)
diff_ci = bootstrap_ci_mean_diff(baseline, ainsf_after)

stats_out = {
    "samples_compared": int(n),
    "baseline_mean": base_mean,
    "baseline_ci95_mean": [base_ci[0], base_ci[1]],
    "ainsf_after_mean": ainsf_mean,
    "ainsf_ci95_mean": [ainsf_ci[0], ainsf_ci[1]],
    "mean_reduction_baseline_minus_ainsf": diff_mean,
    "ci95_mean_reduction": [diff_ci[0], diff_ci[1]],
    "percentiles_baseline": {k: float(np.percentile(baseline, k)) for k in [5, 25, 50, 75, 95]},
    "percentiles_ainsf": {k: float(np.percentile(ainsf_after, k)) for k in [5, 25, 50, 75, 95]},
}

with open(os.path.join(OUT_DIR, "figure_stats.json"), "w", encoding="utf-8") as f:
    json.dump(stats_out, f, indent=2, ensure_ascii=False)
print("✅ Saved:", os.path.join(OUT_DIR, "figure_stats.json"))

print("Samples compared:", n)
print("Baseline mean:", base_mean, "AINSF mean:", ainsf_mean, "Δ:", diff_mean, "CI:", diff_ci)

# =========================================================
# FIG A) ECDF overlay (best for main paper)
# =========================================================
plt.figure(figsize=(6.2, 4.0))
x1, y1 = ecdf(baseline)
x2, y2 = ecdf(ainsf_after)
plt.plot(x1, y1, linewidth=2.0, label="Baseline")
plt.plot(x2, y2, linewidth=2.0, label="AINSF (after)")

# mean markers
plt.axvline(base_mean, linestyle="--", linewidth=1.2)
plt.axvline(ainsf_mean, linestyle="--", linewidth=1.2)

plt.xlabel("Bias probability  P(biased)")
plt.ylabel("ECDF")
plt.title("ECDF of Bias Probability")
plt.legend(frameon=True)
plt.tight_layout()
savefig("figA_ecdf")
plt.close()

# =========================================================
# FIG B) Paired difference histogram (baseline - after)
# =========================================================
plt.figure(figsize=(6.2, 4.0))
bins = np.linspace(float(diff.min()), float(diff.max()), 70)
plt.hist(diff, bins=bins, density=True, alpha=0.9)

plt.axvline(0.0, linestyle="--", linewidth=1.2)
plt.axvline(diff_mean, linestyle="--", linewidth=1.2)

# annotate mean + CI
txt = f"mean Δ = {diff_mean:.3f}\n95% CI [{diff_ci[0]:.3f}, {diff_ci[1]:.3f}]"
plt.text(0.02, 0.95, txt, transform=plt.gca().transAxes, va="top")

plt.xlabel("Paired reduction  (Baseline − AINSF after)")
plt.ylabel("Density")
plt.title("Paired Bias Reduction Distribution")
plt.tight_layout()
savefig("figB_diff_hist")
plt.close()

# =========================================================
# FIG C) Boxplot + sampled points (median + spread)
# =========================================================
plt.figure(figsize=(6.2, 4.0))
plt.boxplot(
    [baseline, ainsf_after],
    labels=["Baseline", "AINSF (after)"],
    showfliers=False,
    widths=0.5,
)

rng = np.random.default_rng(0)
m = min(1200, n)  # fewer points for clean ACL look
idx = rng.choice(np.arange(n), size=m, replace=False)
x_jit1 = 1 + (rng.random(m) - 0.5) * 0.14
x_jit2 = 2 + (rng.random(m) - 0.5) * 0.14
plt.scatter(x_jit1, baseline[idx], alpha=0.18, s=10)
plt.scatter(x_jit2, ainsf_after[idx], alpha=0.18, s=10)

plt.ylabel("Bias probability  P(biased)")
plt.title("Bias Probability: Median and Spread")
plt.tight_layout()
savefig("figC_box_strip")
plt.close()

# =========================================================
# FIG D) Mean with 95% bootstrap CI (simple summary)
# =========================================================
plt.figure(figsize=(6.2, 4.0))
means = [base_mean, ainsf_mean]
errs = [
    [means[0] - base_ci[0], means[1] - ainsf_ci[0]],
    [base_ci[1] - means[0], ainsf_ci[1] - means[1]],
]
x = np.arange(2)

plt.bar(x, means, yerr=errs, capsize=6)
plt.xticks(x, ["Baseline", "AINSF (after)"])
plt.ylabel("Mean  P(biased)")
plt.title("Mean Bias Probability (95% Bootstrap CI)")

# annotate delta
plt.text(0.02, 0.95, f"Δ = {diff_mean:.3f}", transform=plt.gca().transAxes, va="top")

plt.tight_layout()
savefig("figD_mean_ci")
plt.close()

print("\nDONE ✅ ACL-ready figures saved in:", OUT_DIR)

✅ Saved: outputs/figs_acl/figure_stats.json
Samples compared: 5727
Baseline mean: 0.08551742506096563 AINSF mean: 0.06211567507731195 Δ: 0.023401749983653672 CI: (0.01852175449044334, 0.0283852207350657)
✅ Saved: outputs/figs_acl/figA_ecdf.pdf
✅ Saved: outputs/figs_acl/figA_ecdf.png
✅ Saved: outputs/figs_acl/figB_diff_hist.pdf
✅ Saved: outputs/figs_acl/figB_diff_hist.png


/tmp/ipykernel_7440/165602069.py:196: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(


✅ Saved: outputs/figs_acl/figC_box_strip.pdf
✅ Saved: outputs/figs_acl/figC_box_strip.png
✅ Saved: outputs/figs_acl/figD_mean_ci.pdf
✅ Saved: outputs/figs_acl/figD_mean_ci.png

DONE ✅ ACL-ready figures saved in: outputs/figs_acl


In [19]:
#!/usr/bin/env python3
"""
MAKE_HUMAN_EVAL_50.py ✅ (UPDATED for 50)

Changes:
- N_SAMPLES = 50
- If not enough candidates, auto-reduce to max available instead of crashing
- Better match_mode tracking (id vs claim)
- Cleaner final summary print

Outputs:
- outputs/human_eval_50_blind.csv   (share with annotators)
- outputs/human_eval_50_key.json    (DO NOT share; mapping of A/B -> system)
- outputs/human_eval_50_source.jsonl (debug; contains both texts + metadata)

Run:
python3 MAKE_HUMAN_EVAL_50.py
"""

import os, json, csv, re
import numpy as np
from collections import Counter

BASELINE_FILE = "outputs/baseline_rag_llm_biasgate.jsonl"
AINSF_FILE    = "outputs/ainsf_pass_final_with_after.jsonl"

OUT_CSV = "outputs/human_eval_50_blind.csv"
OUT_KEY = "outputs/human_eval_50_key.json"
OUT_SRC = "outputs/human_eval_50_source.jsonl"

N_SAMPLES = 50
SEED = 42

# Which AINSF rows to evaluate (rewritten only)
REWRITE_STATUSES = {"OK", "RETRIED_OK", "FALLBACK"}

os.makedirs("outputs", exist_ok=True)

ID_CANDIDATES = ["id", "claim_id", "uid", "uuid", "idx", "example_id", "row_id", "sample_id"]
CLAIM_CANDIDATES = ["claim", "text", "input", "original_claim", "prompt", "source"]

def read_jsonl(path):
    rows = []
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def pick_first_key(row, candidates):
    for k in candidates:
        if k in row and row[k] is not None:
            return k
    return None

def safe_id(x):
    return str(x) if x is not None else None

def norm_text(s: str) -> str:
    if s is None:
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def get_text(row):
    """
    Extract baseline output text.
    Your baseline file stores it in: rag_output
    """
    v = row.get("rag_output", None)
    if isinstance(v, str) and v.strip():
        return v.strip()

    # other common fallbacks
    for k in ["final_text", "output_text", "generated_text", "prediction", "text", "answer", "response", "output"]:
        v = row.get(k, None)
        if isinstance(v, str) and v.strip():
            return v.strip()

    # nested fallbacks
    for path in [
        ("rag", "final_text"),
        ("rag", "answer"),
        ("generation", "text"),
    ]:
        cur = row
        ok = True
        for p in path:
            if isinstance(cur, dict) and p in cur:
                cur = cur[p]
            else:
                ok = False
                break
        if ok and isinstance(cur, str) and cur.strip():
            return cur.strip()

    return None

def get_claim(row, claim_key):
    if claim_key and isinstance(row.get(claim_key), str) and row[claim_key].strip():
        return row[claim_key].strip()
    return ""

# -------------------------
# Load data
# -------------------------
baseline_rows = read_jsonl(BASELINE_FILE)
ainsf_rows    = read_jsonl(AINSF_FILE)

if not baseline_rows or not ainsf_rows:
    raise RuntimeError("One of the input files is empty.")

# Detect likely keys
base_id_key  = pick_first_key(baseline_rows[0], ID_CANDIDATES)
ainsf_id_key = pick_first_key(ainsf_rows[0], ID_CANDIDATES)

base_claim_key  = pick_first_key(baseline_rows[0], CLAIM_CANDIDATES)
ainsf_claim_key = pick_first_key(ainsf_rows[0], CLAIM_CANDIDATES)

print("Detected keys:")
print("  baseline id key :", base_id_key)
print("  AINSF id key    :", ainsf_id_key)
print("  baseline claim  :", base_claim_key)
print("  AINSF claim     :", ainsf_claim_key)
print("  rewrite_status  : uses r.get('rewrite_status')")

# -------------------------
# Build baseline indices
# -------------------------
baseline_by_id = {}        # id -> {row, text}
baseline_by_claim = {}     # norm(claim) -> {row, text}

missing_base_id = 0
missing_base_txt = 0
missing_base_claim = 0

for r in baseline_rows:
    rid = safe_id(r.get(base_id_key)) if base_id_key else None
    txt = get_text(r)
    clm = get_claim(r, base_claim_key)

    if not txt:
        missing_base_txt += 1
        continue

    if rid is None:
        missing_base_id += 1
    else:
        baseline_by_id[rid] = {"row": r, "text": txt}

    if not clm:
        missing_base_claim += 1
    else:
        baseline_by_claim[norm_text(clm)] = {"row": r, "text": txt}

print("\nBaseline index stats:")
print("  total baseline rows:", len(baseline_rows))
print("  baseline_by_id size:", len(baseline_by_id), "| missing id rows:", missing_base_id)
print("  baseline_by_claim size:", len(baseline_by_claim), "| missing claim rows:", missing_base_claim)
print("  missing baseline text rows:", missing_base_txt)

# -------------------------
# Candidate selection (AINSF rewritten rows only)
# -------------------------
candidate = []
no_status = 0
no_ainsf_text = 0
no_match = 0
match_by_id = 0
match_by_claim = 0

status_counts = Counter([(r.get("rewrite_status") or "").upper() for r in ainsf_rows])
print("\nTop rewrite_status values in AINSF:")
print(status_counts.most_common(15))

for r in ainsf_rows:
    st = (r.get("rewrite_status") or "").upper()
    if not st:
        no_status += 1
        continue
    if st not in REWRITE_STATUSES:
        continue

    ainsf_txt = r.get("final_text")
    if not isinstance(ainsf_txt, str) or not ainsf_txt.strip():
        no_ainsf_text += 1
        continue

    match_mode = None
    base_hit = None

    # 1) Try ID match
    rid_ainsf = safe_id(r.get(ainsf_id_key)) if ainsf_id_key else None
    if rid_ainsf is not None and rid_ainsf in baseline_by_id:
        base_hit = baseline_by_id[rid_ainsf]
        match_by_id += 1
        match_mode = "id"
    else:
        # 2) fallback: claim match
        ainsf_claim = get_claim(r, ainsf_claim_key)
        if ainsf_claim:
            base_hit = baseline_by_claim.get(norm_text(ainsf_claim))
            if base_hit is not None:
                match_by_claim += 1
                match_mode = "claim"

    if base_hit is None:
        no_match += 1
        continue

    candidate.append({"ainsf": r, "baseline": base_hit, "match_mode": match_mode})

print("\nCandidate stats:")
print("  candidates:", len(candidate))
print("  match_by_id:", match_by_id, "| match_by_claim:", match_by_claim)
print("  rewritten rows missing status:", no_status)
print("  rewritten rows missing final_text:", no_ainsf_text)
print("  rewritten rows with no baseline match:", no_match)

# ✅ Auto-adjust sample size if limited
if len(candidate) == 0:
    raise RuntimeError("No candidates found. Check rewrite_status values and keys.")

final_n = min(N_SAMPLES, len(candidate))
if final_n < N_SAMPLES:
    print(f"\n⚠️ Not enough candidates for {N_SAMPLES}. Will create {final_n} samples instead.")

rng = np.random.default_rng(SEED)
picked_idx = rng.choice(len(candidate), size=final_n, replace=False)
picked = [candidate[i] for i in picked_idx]

# -------------------------
# Build blind CSV + secret key
# -------------------------
blind_rows = []
key = {
    "seed": SEED,
    "n_samples": final_n,
    "baseline_file": BASELINE_FILE,
    "ainsf_file": AINSF_FILE,
    "note": "A/B order randomized per item. Do NOT share this key with annotators.",
    "detected_keys": {
        "baseline_id_key": base_id_key,
        "ainsf_id_key": ainsf_id_key,
        "baseline_claim_key": base_claim_key,
        "ainsf_claim_key": ainsf_claim_key,
    },
    "items": []
}

with open(OUT_SRC, "w", encoding="utf-8") as fsrc:
    for i, pack in enumerate(picked, start=1):
        r = pack["ainsf"]
        b = pack["baseline"]
        match_mode = pack["match_mode"]

        # Prefer stable sample_id
        rid = safe_id(r.get(ainsf_id_key)) if ainsf_id_key else None
        if rid is None:
            rid = norm_text(get_claim(r, ainsf_claim_key))[:80] or f"row_{i}"

        base_txt = b["text"].strip()
        ainsf_txt = r["final_text"].strip()

        # Randomize A/B
        if rng.random() < 0.5:
            A_txt, B_txt = base_txt, ainsf_txt
            A_sys, B_sys = "BASELINE", "AINSF"
        else:
            A_txt, B_txt = ainsf_txt, base_txt
            A_sys, B_sys = "AINSF", "BASELINE"

        claim = get_claim(r, ainsf_claim_key) or get_claim(b["row"], base_claim_key) or ""
        claim = claim.strip()

        blind_rows.append({
            "item_id": i,
            "sample_id": rid,
            "claim": claim,
            "Text_A": A_txt,
            "Text_B": B_txt,
            "more_neutral": "",       # A / B / Equal
            "meaning_preserved": "",  # A / B / Both / Neither
            "notes": ""
        })

        key["items"].append({
            "item_id": i,
            "sample_id": rid,
            "A_system": A_sys,
            "B_system": B_sys,
            "ainsf_rewrite_status": (r.get("rewrite_status") or "").upper(),
            "match_mode": match_mode
        })

        fsrc.write(json.dumps({
            "item_id": i,
            "sample_id": rid,
            "claim": claim,
            "baseline_text": base_txt,
            "ainsf_text": ainsf_txt,
            "ainsf_rewrite_status": (r.get("rewrite_status") or "").upper(),
            "match_mode": match_mode,
            "detected": {
                "baseline_id_key": base_id_key,
                "ainsf_id_key": ainsf_id_key,
                "baseline_claim_key": base_claim_key,
                "ainsf_claim_key": ainsf_claim_key,
            }
        }, ensure_ascii=False) + "\n")

# Write CSV
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(blind_rows[0].keys()))
    w.writeheader()
    w.writerows(blind_rows)

# Write key
with open(OUT_KEY, "w", encoding="utf-8") as f:
    json.dump(key, f, indent=2, ensure_ascii=False)

print("\n✅ Created blind human eval set:")
print("   CSV (share):", OUT_CSV)
print("   KEY (private):", OUT_KEY)
print("   SOURCE (debug):", OUT_SRC)
print(f"   Final sample count: {final_n}")
print("Tip: Give CSV to 2–3 annotators. Keep KEY with you.")

Detected keys:
  baseline id key : id
  AINSF id key    : id
  baseline claim  : claim
  AINSF claim     : claim
  rewrite_status  : uses r.get('rewrite_status')

Baseline index stats:
  total baseline rows: 10000
  baseline_by_id size: 10000 | missing id rows: 0
  baseline_by_claim size: 9743 | missing claim rows: 0
  missing baseline text rows: 0

Top rewrite_status values in AINSF:
[('NOT_BIASED', 5584), ('FALLBACK', 74), ('OK', 45), ('RETRIED_OK', 24)]

Candidate stats:
  candidates: 143
  match_by_id: 143 | match_by_claim: 0
  rewritten rows missing status: 0
  rewritten rows missing final_text: 0
  rewritten rows with no baseline match: 0

✅ Created blind human eval set:
   CSV (share): outputs/human_eval_50_blind.csv
   KEY (private): outputs/human_eval_50_key.json
   SOURCE (debug): outputs/human_eval_50_source.jsonl
   Final sample count: 50
Tip: Give CSV to 2–3 annotators. Keep KEY with you.
